# Phase 07 - Analysis

Aggregation over splits, tables, figures, Grad-CAM and failure cases.
See `docs/instructions/07-analysis.md`.

Cells whose first line is `#| export` are copied, in order, into the
generated `group3-starvenus.py` by `tools/export_py.py`. Keep outputs cleared before committing.

## Aggregation core (T7.1)

`load_experiment` reads `_work/runs/<run folder>/split_0..4/metrics.json` and accepts an experiment only if all
five splits are present, each run's split sha256 equals the committed `_work/splits/split_<k>.json`, and its
config fingerprint equals the one the current registry produces (so a stale run, or a registry edit that would
make `run` retrain, is reported instead of silently mixed into a table). Anything else is skipped with a printed
warning. Non-directories such as `runs/env.json`, and the `sweep/` and `scratch/` folders, are never read as results.

* `aggregate(exp, subset, metric, part="test")` -> `Aggregate(mean, std, values)`; std is the sample std (`ddof=1`).
* `paired_difference(a, b, subset, metric, part)` -> mean +- std of the per-split difference a - b, plus the number
  of splits in which a is ahead. All experiments share the same five splits, so split-to-split variation cancels.
* Number format: macro-F1 with 3 decimals, accuracy in % with 1 decimal, always `mean +- std`.

In [ ]:
#| export
import re
from typing import NamedTuple

ANALYSIS_SPLITS = (0, 1, 2, 3, 4)  # every reported number is a mean +- std over these five splits
RETRAIN_RUNS_FOLDER = "retrain"  # runs/retrain/<run folder>/split_k/: the retrained copies of the Grad-CAM models, with best.pt
IGNORED_RUN_FOLDERS = ("sweep", "scratch", RETRAIN_RUNS_FOLDER)  # under runs/: learning-rate sweeps, dev runs and retrained copies (Grad-CAM), never results
SCORE_SUBSETS = ("white", "field", "mixed", "domain_balanced")
SCORE_METRICS = ("macro_f1", "accuracy")
PARTS = ("val", "test")
SUBSET_LABELS = {"white": "White", "field": "Field", "mixed": "Mixed", "domain_balanced": "Domain-balanced"}
METRIC_LABELS = {"macro_f1": "macro-F1", "accuracy": "accuracy (%)"}
# Rows of the required tier that the tables cover (the later phases add their own rows).
ANALYSIS_EXPERIMENTS = (
    "E0-squash", "E0-pad", "E0-rect", "E1-U", "E1-WW", "E1-FF", "E1-WF",
    "E2-cnx-U", "E2-cnx-WF", "E2-swin-U", "E2-swin-WF",
)


@dataclass(frozen=True)
class ExperimentResults:
    """The complete, validated metrics of one experiment over the five splits."""

    exp_id: str  # the requested ID (E1-U for the alias)
    run_name: str  # folder under runs/ (E0-rect for E1-U)
    config: ExperimentConfig  # the registry's resolved config; its fingerprint matched every split
    metrics: dict  # split seed -> metrics.json content

    @property
    def splits(self) -> tuple:
        return tuple(sorted(self.metrics))


def expected_fingerprint(config: ExperimentConfig, composite_factor: float | None = None, data_root=None) -> str:
    """The fingerprint a plain `run` of this config writes (no seed offset, not --fast). Method A rows also carry their
    composites-per-epoch factor and the composite recipe (``composite_fingerprint_extra``), the segmenter and Method B
    rows their recipes (``seg_fingerprint_extra``; the segmenter's needs ``data_root`` for the reject list); other rows
    are unchanged."""
    return config_fingerprint(config, {"seed_offset": 0, "limit": None, **composite_fingerprint_extra(config, composite_factor),
                                       **seg_fingerprint_extra(config, data_root)})


def boxes_problem(data_root, split_seed: int, metrics: dict) -> str | None:
    """Method B runs only: why the split's run does not match the current S-seg boxes.json (None when it does)."""
    path = boxes_path(data_root, split_seed)
    if not path.is_file():
        return f"split {split_seed}: {path} is missing, so the boxes this run cropped by cannot be verified"
    if metrics.get("seg_boxes_sha256") != file_sha256(path):
        return f"split {split_seed}: the S-seg boxes changed since this run (boxes.json sha256 differs; stale run)"
    return None


def load_experiment(data_root, exp_id: str, choices) -> tuple:
    """(ExperimentResults, []) for a complete experiment whose runs match the committed splits and the registry,
    otherwise (None, [reasons]). Never raises for an experiment that is merely not run yet."""
    try:
        resolved = resolve_experiment(exp_id, choices)
        run_name = run_folder_name(exp_id, choices)
    except ExperimentError as exc:
        needed = re.search(r"needed: ([^)]*)\)", str(exc))
        return None, [f"not ready, choices.json has no {needed.group(1)} yet" if needed else f"not ready ({exc})"]
    run_root = runs_dir(data_root) / run_name
    if not run_root.is_dir():
        return None, [f"pending, no runs yet ({run_root} does not exist)"]
    expected = expected_fingerprint(resolved.config, resolved.composite_factor, data_root)
    problems, found = [], {}
    for split_seed in ANALYSIS_SPLITS:
        metrics = read_metrics(split_dir(run_root, split_seed))
        if metrics is None:
            problems.append(f"split {split_seed}: no metrics.json (run incomplete)")
            continue
        try:
            committed_sha = load_split(data_root, split_seed)["sha256"]
        except DatasetError as exc:
            problems.append(f"split {split_seed}: {exc}")
            continue
        if metrics.get("split_sha256") != committed_sha:
            problems.append(f"split {split_seed}: split sha256 differs from the committed split (stale run)")
        elif metrics.get("config_fingerprint") != expected:
            problems.append(f"split {split_seed}: config fingerprint differs from the registry (the experiment definition changed)")
        elif metrics.get("fast") or metrics.get("overrides"):
            problems.append(f"split {split_seed}: this is a --fast / dev-override run, not a result")
        elif resolved.config.seg_crop and boxes_problem(data_root, split_seed, metrics):
            problems.append(boxes_problem(data_root, split_seed, metrics))
        else:
            found[split_seed] = metrics
    if problems:
        return None, problems
    return ExperimentResults(exp_id, run_name, resolved.config, found), []


def load_experiments(data_root, exp_ids, choices) -> dict:
    """{exp_id: ExperimentResults} for the complete experiments. Every other one is skipped with a printed warning."""
    results = {}
    for exp_id in exp_ids:
        found, problems = load_experiment(data_root, exp_id, choices)
        if found is None:
            warn(f"{exp_id}: skipped - " + "; ".join(problems))
        else:
            results[exp_id] = found
    known = set()
    for exp_id in EXPERIMENTS:
        try:
            known.add(run_folder_name(exp_id, choices))
        except ExperimentError:
            known.add(exp_id)
    runs_root = runs_dir(data_root)
    for folder in sorted(runs_root.iterdir()) if runs_root.is_dir() else ():
        if folder.is_dir() and folder.name not in IGNORED_RUN_FOLDERS and folder.name not in known:
            warn(f"{folder}: not a run folder of any registry experiment, ignored")
    return results


def mean_std(values) -> tuple:
    """(mean, sample std with ddof=1). NaN when a value is missing or there are fewer than two values."""
    if len(values) == 0 or any(v is None for v in values):
        return float("nan"), float("nan")
    array = np.asarray(values, dtype=np.float64)
    std = float(np.std(array, ddof=1)) if len(array) > 1 else float("nan")
    return float(np.mean(array)), std


class Aggregate(NamedTuple):
    mean: float
    std: float  # ddof=1 (sample std over the splits)
    values: list  # per-split values, in split order


class PairedDifference(NamedTuple):
    mean: float
    std: float
    values: list  # a - b for each split
    ahead: int  # number of splits in which a is strictly ahead of b


def split_values(exp: ExperimentResults, subset: str, metric: str, part: str) -> list:
    if subset not in SCORE_SUBSETS or metric not in SCORE_METRICS or part not in PARTS:
        raise ValueError(f"unknown subset/metric/part: {subset!r}, {metric!r}, {part!r}")
    return [exp.metrics[k][part][subset][metric] for k in exp.splits]


def aggregate(exp: ExperimentResults, subset: str, metric: str, part: str = "test") -> Aggregate:
    """Mean, std (ddof=1) and per-split list of one metric of one experiment."""
    values = split_values(exp, subset, metric, part)
    mean, std = mean_std(values)
    return Aggregate(mean, std, values)


def paired_difference(a: ExperimentResults, b: ExperimentResults, subset: str, metric: str, part: str = "test") -> PairedDifference:
    """Per-split difference a - b (all experiments share the same five splits), as mean +- std."""
    if a.splits != b.splits:
        raise ValueError(f"{a.exp_id} and {b.exp_id} do not share the same splits: {a.splits} vs {b.splits}")
    diffs = [x - y for x, y in zip(split_values(a, subset, metric, part), split_values(b, subset, metric, part))]
    mean, std = mean_std(diffs)
    return PairedDifference(mean, std, diffs, sum(d > 0 for d in diffs))


def format_mean_std(stat, metric: str, *, signed: bool = False) -> str:
    """'0.912 +- 0.014' for F1 (3 decimals), '91.2 +- 1.4' for accuracy (percent, 1 decimal); signed for differences."""
    if stat.mean != stat.mean or stat.std != stat.std:
        return "n/a"
    scale, decimals = (100.0, 1) if metric == "accuracy" else (1.0, 3)
    sign = "+" if signed else ""
    return f"{stat.mean * scale:{sign}.{decimals}f} +- {stat.std * scale:.{decimals}f}"

## Tables (T7.2)

Every table is written as `.csv` (for Word) and `.md` (for review). Rows of experiments that are not complete yet
read `pending`, so the skeletons fill in as runs arrive.

In [ ]:
#| export
@dataclass(frozen=True)
class Table:
    """A results table, written as <name>.csv (for Word) and <name>.md (for review in the repo)."""

    name: str
    title: str
    header: tuple
    rows: list
    note: str = ""

    def to_csv(self) -> str:
        return csv_text(list(self.header), self.rows)

    def to_markdown(self) -> str:
        lines = [f"# {self.title}", ""]
        if self.note:
            lines += [self.note, ""]
        lines.append("| " + " | ".join(self.header) + " |")
        lines.append("|" + "|".join(" --- " for _ in self.header) + "|")
        lines += ["| " + " | ".join(str(cell) for cell in row) + " |" for row in self.rows]
        return "\n".join(lines) + "\n"


def write_table(data_root, table: Table) -> list:
    paths = []
    for suffix, text in ((".csv", table.to_csv()), (".md", table.to_markdown())):
        path = results_dir(data_root) / (table.name + suffix)
        write_text_atomic(path, text)
        paths.append(path)
    return paths


PENDING = "pending"
SCORE_HEADER = tuple(f"{SUBSET_LABELS[s]} {METRIC_LABELS[m]}" for s in SCORE_SUBSETS for m in SCORE_METRICS)


def score_cells(exp, part: str, subsets=SCORE_SUBSETS) -> list:
    """White / field / mixed / domain-balanced, each as macro-F1 and accuracy (mean +- std). '-' for subsets a row
    does not report, 'pending' for an experiment that is not complete yet."""
    if exp is None:
        return [PENDING] * len(SCORE_HEADER)
    return [
        format_mean_std(aggregate(exp, s, m, part), m) if s in subsets else "-"
        for s in SCORE_SUBSETS
        for m in SCORE_METRICS
    ]


def difference_cells(a, b, part: str) -> list:
    if a is None or b is None:
        return [PENDING] * len(SCORE_HEADER)
    return [format_mean_std(paired_difference(a, b, s, m, part), m, signed=True) for s in SCORE_SUBSETS for m in SCORE_METRICS]


def table_input_mode(results: dict) -> Table:
    """E0 (G0 evidence): ResNet50 on white + field at three input sizes, val and test, and the paired
    differences to rect."""
    rows = []
    for part in PARTS:
        for mode in INPUT_MODES:
            height, width = INPUT_SIZES[mode]
            rows.append([part, mode, f"{height}x{width}", *score_cells(results.get(f"E0-{mode}"), part)])
        for mode in INPUT_MODES:
            if mode != "rect":
                rows.append([part, f"{mode} - rect (paired)", "-", *difference_cells(results.get(f"E0-{mode}"), results.get("E0-rect"), part)])
    return Table(
        "table_input_mode",
        "Input mode ablation (E0, ResNet50, white + field training)",
        ("Part", "Input mode", "Input HxW", *SCORE_HEADER),
        rows,
        "Mean +- std over the 5 splits (std with ddof=1; macro-F1 with 3 decimals, accuracy in % with 1 decimal). "
        "Gate G0 used only the val rows. The paired rows are the per-split difference of the first mode minus rect. "
        "Mixed is the full split (about 70% white); domain-balanced is the mean of white and field.",
    )


def table_required(results: dict, choices=None) -> Table:
    """The unified models on the test split (spec scenarios a, b, c plus the domain-balanced mean). The recommended method
    (gate G2) is marked, and added as a row when it is not one of the three plain models."""
    recommended = recommended_experiment(choices) if choices is not None else None
    rows = []
    for exp_id in dict.fromkeys(("E1-U", "E2-cnx-U", "E2-swin-U", *([recommended] if recommended else []))):
        found = results.get(exp_id)
        backbone = found.config.backbone if found is not None else EXPERIMENTS[exp_id].backbone
        rows.append([exp_id + (" (recommended)" if exp_id == recommended else ""), backbone, *score_cells(found, "test")])
    return Table(
        "table_required",
        "Required results: unified models, test split (E1-U, E2-cnx-U, E2-swin-U)",
        ("Experiment", "Backbone", *SCORE_HEADER),
        rows,
        "Mean +- std over the 5 splits (std with ddof=1). White, field and mixed are the three spec scenarios; "
        "domain-balanced is the mean of white and field. The recommended method is the one gate G2 froze. "
        "Rows that are not complete yet read 'pending'.",
    )


DOMAIN_EXPERIMENTS = ("E1-WW", "E1-FF", "E1-WF", "E1-U", "E2-cnx-WF", "E2-swin-WF")
OPTIONAL_DOMAIN_EXPERIMENTS = ("E3-A-WF",)  # Tier 4: a row only once it has run


def table_domains(results: dict, choices) -> Table:
    """Domain-wise and cross-domain experiments on the test split (only the subsets each row is about). E3-A-WF is a
    Method A test number, so like the test rows of table_methods it stays hidden until gate G2 froze the recommendation."""
    recommended = choices.raw("recommended")
    rows = []
    for exp_id in (*DOMAIN_EXPERIMENTS, *(e for e in OPTIONAL_DOMAIN_EXPERIMENTS if e in results)):
        base = EXPERIMENTS[exp_id]
        found = results.get(exp_id)
        backbone = found.config.backbone if found is not None else base.backbone
        train_on = " + ".join(base.train_domains) + (" + composites" if base.use_composites else "")
        subsets = set(base.test_subsets) | ({"domain_balanced"} if "mixed" in base.test_subsets else set())
        cells = score_cells(found, "test", subsets)
        if base.use_composites and recommended is None:
            cells = [HIDDEN_UNTIL_G2 if cell != "-" else cell for cell in cells]
        rows.append([exp_id, backbone, train_on, *cells])
    return Table(
        "table_domains",
        "Domain-wise and cross-domain experiments, test split",
        ("Experiment", "Backbone", "Train on", *SCORE_HEADER),
        rows,
        "W->W is E1-WW, F->F is E1-FF, W->F is E1-WF (field is its main result, white the reference), U is E1-U. "
        "E3-A-WF (when run) trains on white images plus composites: field images are only their backgrounds, so it is not "
        "pure W->F; its test numbers are hidden until gate G2. '-' marks subsets that a row is not about. Mean +- std over "
        "the 5 splits. Rows that are not complete yet read 'pending'.",
    )

## Gates G0 and G1 (phase 04)

`gate g0` and `gate g1` recompute the decision from the run folders (never from typed-in numbers) and write the
entry to `_work/results/choices.json` with its evidence: per-candidate mean, std and per-split values, the paired
differences, the rule text and the clause that decided. Other keys of `choices.json` are left alone, so the marker's
`reproduce` can re-derive every choice. Gates use **validation** metrics only.

* **G0**: highest mean val mixed macro-F1 among squash / pad / rect. If the top two are within one std, prefer `rect`.
* **G1**: highest mean val mixed macro-F1 among ResNet50 (E1-U), ConvNeXt-T, Swin-T. If the top two are within one
  std, prefer the higher mean val field macro-F1. The winner's learning rate is `lr[backbone]`, already in `choices.json`
  from the sweeps, and is copied into the evidence.
* "Within one std" means the gap between the two means is at most the larger of the two stds (symmetric in the pair).

In [ ]:
#| export
GATE_METRIC = "validation mixed macro-F1 (mean over the 5 splits; std with ddof=1)"
TIE_RULE = "The top two are 'within one std' when their gap in mean is at most the larger of their two stds."
G0_RULE = (
    f"G0 (phase 04): choose the input mode with the highest {GATE_METRIC}. If the top two are within one std "
    "of each other, prefer rect (it preserves leaf geometry at no extra cost). " + TIE_RULE
)
G1_RULE = (
    f"G1 (phase 04): choose the backbone with the highest {GATE_METRIC}, among ResNet50 (E1-U), ConvNeXt-T and "
    "Swin-T. If the top two are within one std of each other, prefer the one with the higher mean validation "
    "field macro-F1 (field is the harder, more valuable domain). " + TIE_RULE
)
G1_CANDIDATES = {"resnet50": "E1-U", "convnext_tiny": "E2-cnx-U", "swin_t": "E2-swin-U"}


def rank_candidates(mixed: dict) -> list:
    """Candidate names by decreasing mean (a tie keeps the order of ``mixed``)."""
    return sorted(mixed, key=lambda name: -mixed[name].mean)


def top_two_gap(mixed: dict, ranked: list) -> tuple:
    """(gap between the top two means, tolerance of one std, whether the gap is within it)."""
    first, second = mixed[ranked[0]], mixed[ranked[1]]
    gap = first.mean - second.mean
    tolerance = max(first.std, second.std)
    return gap, tolerance, gap <= tolerance


def decide_g0(mixed: dict) -> dict:
    """Gate G0. ``mixed`` maps input mode -> Aggregate of val mixed macro-F1. Returns the winner and the clause."""
    ranked = rank_candidates(mixed)
    gap, tolerance, tied = top_two_gap(mixed, ranked)
    if tied and ranked[0] != "rect" and "rect" in ranked[:2]:
        value, clause = "rect", "within one std -> rect"
    elif tied and "rect" not in ranked[:2]:
        value, clause = ranked[0], "highest mean (the top two are within one std but rect is not one of them)"
    else:
        value, clause = ranked[0], "highest mean"
    return {"value": value, "decided_by": clause, "ranking": ranked, "top_two_gap": gap, "tolerance": tolerance}


def decide_g1(mixed: dict, field: dict) -> dict:
    """Gate G1. ``mixed`` / ``field`` map backbone -> Aggregate of val mixed / val field macro-F1."""
    ranked = rank_candidates(mixed)
    gap, tolerance, tied = top_two_gap(mixed, ranked)
    if tied:
        value = max(ranked[:2], key=lambda name: (field[name].mean, -ranked.index(name)))
        clause = "within one std -> higher val field macro-F1"
    else:
        value, clause = ranked[0], "highest mean"
    return {"value": value, "decided_by": clause, "ranking": ranked, "top_two_gap": gap, "tolerance": tolerance}


def stat_record(stat) -> dict:
    record = {"mean": float(stat.mean), "std": float(stat.std), "per_split": [float(v) for v in stat.values]}
    if isinstance(stat, PairedDifference):
        record["splits_ahead"] = int(stat.ahead)
    return record


def require_complete(data_root, exp_id: str, choices, gate: str) -> ExperimentResults:
    found, problems = load_experiment(data_root, exp_id, choices)
    if found is None:
        raise ExperimentError(f"gate {gate}: {exp_id} cannot be used - " + "; ".join(problems))
    return found


def candidate_evidence(exps: dict) -> dict:
    out = {}
    for name, exp in exps.items():
        out[name] = {
            "experiment": exp.exp_id, "run_folder": exp.run_name,
            "config_fingerprint": exp.metrics[exp.splits[0]]["config_fingerprint"],
            "split_sha256": [exp.metrics[k]["split_sha256"] for k in exp.splits],
            "val_mixed_macro_f1": stat_record(aggregate(exp, "mixed", "macro_f1", "val")),
            "val_field_macro_f1": stat_record(aggregate(exp, "field", "macro_f1", "val")),
        }
    return out


def paired_evidence(exps: dict, pairs) -> dict:
    return {
        f"{a}_minus_{b}": {
            "val_mixed_macro_f1": stat_record(paired_difference(exps[a], exps[b], "mixed", "macro_f1", "val")),
            "val_field_macro_f1": stat_record(paired_difference(exps[a], exps[b], "field", "macro_f1", "val")),
        }
        for a, b in pairs
    }


def gate_evidence(rule: str, decision: dict, exps: dict, pairs, **extra) -> dict:
    return {
        "rule": rule, "metric": GATE_METRIC, "decided_by": decision["decided_by"], "ranking": decision["ranking"],
        "top_two_gap": float(decision["top_two_gap"]), "tolerance_one_std": float(decision["tolerance"]), **extra,
        "candidates": candidate_evidence(exps), "paired_differences": paired_evidence(exps, pairs),
    }


def evaluate_g0(exps: dict) -> dict:
    """{"value", "evidence"} of gate G0 from the three complete E0 experiments (keyed by input mode)."""
    decision = decide_g0({mode: aggregate(exps[mode], "mixed", "macro_f1", "val") for mode in INPUT_MODES})
    pairs = [(mode, "rect") for mode in INPUT_MODES if mode != "rect"]
    return {"value": decision["value"], "evidence": gate_evidence(G0_RULE, decision, exps, pairs)}


def evaluate_g1(exps: dict, lrs: dict) -> dict:
    """{"value", "evidence"} of gate G1 from the three complete unified experiments (keyed by backbone);
    ``lrs`` maps backbone -> its learning rate from choices.json."""
    mixed = {name: aggregate(exps[name], "mixed", "macro_f1", "val") for name in G1_CANDIDATES}
    field = {name: aggregate(exps[name], "field", "macro_f1", "val") for name in G1_CANDIDATES}
    decision = decide_g1(mixed, field)
    pairs = [("convnext_tiny", "resnet50"), ("swin_t", "resnet50"), ("swin_t", "convnext_tiny")]
    winner = decision["value"]
    lr = {"backbone": winner, "value": lrs[winner], "source": "choices.json lr[backbone], chosen by the split-0 sweep"}
    return {"value": winner, "evidence": gate_evidence(G1_RULE, decision, exps, pairs, lr=lr)}


def describe_gate(gate: str, result: dict) -> str:
    ev = result["evidence"]

    def cell(record: dict) -> str:
        return f"{record['mean']:.3f} +- {record['std']:.3f}"

    lines = [f"gate {gate}: {ev['metric']}", f"  {'candidate':<15}{'val mixed F1':>16}{'val field F1':>16}"]
    for name in ev["ranking"]:
        c = ev["candidates"][name]
        lines.append(f"  {name:<15}{cell(c['val_mixed_macro_f1']):>16}{cell(c['val_field_macro_f1']):>16}")
    first, second = ev["ranking"][:2]
    within = "within" if ev["top_two_gap"] <= ev["tolerance_one_std"] else "outside"
    lines.append(f"  top two: {first} leads {second} by {ev['top_two_gap']:.3f}; one std = {ev['tolerance_one_std']:.3f} ({within} one std)")
    lines.append(f"  decision: {result['value']}  ({ev['decided_by']})")
    return "\n".join(lines)


def run_gate(data_root, gate: str, out=print, *, cut=(), force: bool = False) -> dict:
    """Recompute gate ``g0``, ``g1`` or ``g2`` from the run folders and write its entry to choices.json (other keys
    stay). ``cut`` and ``force`` belong to g2 (see its section below)."""
    data_root = Path(data_root)
    choices = load_choices(data_root)
    if gate != "g2" and (cut or force):
        raise ExperimentError(f"gate {gate}: --cut and --force belong to gate g2")
    if gate == "g2":
        return run_gate_g2(data_root, choices, cut=tuple(cut), force=force, out=out)
    if gate == "g0":
        exps = {mode: require_complete(data_root, f"E0-{mode}", choices, gate) for mode in INPUT_MODES}
        result, key = evaluate_g0(exps), "input_mode"
    elif gate == "g1":
        exps = {name: require_complete(data_root, exp_id, choices, gate) for name, exp_id in G1_CANDIDATES.items()}
        result, key = evaluate_g1(exps, {name: choices.get("lr", name) for name in G1_CANDIDATES}), "backbone"
    else:
        raise ExperimentError(f"unknown gate {gate!r}, expected g0, g1 or g2")
    path = write_choice(data_root, key, result["value"], result["evidence"])
    out(describe_gate(gate, result))
    out(f"  written to {path}: {key} = {result['value']!r}" + (f", lr = {result['evidence']['lr']['value']:g}" if gate == "g1" else ""))
    return result

## Gate G2: the recommended method (phase 06, T6.4)

`gate g2` chooses the method the report recommends, from three candidates: the **plain** unified model of the G1
backbone (`E1-U`, `E2-cnx-U` or `E2-swin-U`), **Method A** (`E3-A`, composites) and **Method B** (`E4-B`, segment then crop).
Like G0 and G1 it reads only **validation** metrics from the run folders.

* Highest mean val mixed macro-F1. If the top two are within one std, the higher mean val field macro-F1 wins, unless the
  two field means are also within one std of each other: then the simpler of the two wins (plain < A < B).
* Every candidate must be complete (5 splits, current fingerprint). A method that was dropped by a cut line of the README
  is left out only when it is named explicitly: `gate g2 --cut B` (or `--cut A --cut B`); the evidence records the cut.
  The plain model can never be cut.
* The result is written as `choices.json["recommended"] = {"value": <experiment ID>, "evidence": {...}}`, the ID that
  `predict` uses by default. **It is frozen**: running `gate g2` again with a different outcome refuses unless `--force` is
  given (06: flipping the choice after the test numbers are seen is test-set selection). Until `recommended` exists,
  `table_methods` shows validation numbers only.

In [ ]:
#| export
G2_METHODS = ("plain", "A", "B")  # simplest first: the last tie-break prefers the earlier one
G2_METHOD_EXPERIMENTS = {"A": "E3-A", "B": "E4-B"}
G2_METHOD_LABELS = {"plain": "plain unified model", "A": "Method A (composites)", "B": "Method B (segment, crop)"}
G2_RULE = (
    f"G2 (phase 06): recommend the method with the highest {GATE_METRIC}, among the plain unified model of the G1 "
    "backbone, Method A (E3-A) and Method B (E4-B). If the top two are within one std of each other, prefer the one with "
    "the higher mean validation field macro-F1; if their field means are also within one std, prefer the simpler method "
    "(plain < A < B). " + TIE_RULE
)


def within_one_std(a, b) -> tuple:
    """(|gap| of the two means, tolerance = the larger std, whether the gap is within it)."""
    gap, tolerance = abs(a.mean - b.mean), max(a.std, b.std)
    return gap, tolerance, gap <= tolerance


def decide_g2(mixed: dict, field: dict) -> dict:
    """Gate G2. ``mixed`` / ``field`` map method ("plain", "A", "B"; the ones not cut) -> Aggregate of val mixed / field
    macro-F1. A tie in mean keeps the G2_METHODS order, so the simpler method ranks first."""
    methods = [m for m in G2_METHODS if m in mixed]
    ranked = sorted(methods, key=lambda m: -mixed[m].mean)
    if len(ranked) == 1:
        return {"value": ranked[0], "decided_by": "only candidate (the others were cut)", "ranking": ranked,
                "top_two_gap": None, "tolerance": None, "field_gap": None, "field_tolerance": None}
    gap, tolerance, tied = top_two_gap(mixed, ranked)
    first, second = ranked[:2]
    field_gap = field_tolerance = None
    if not tied:
        value, clause = first, "highest mean"
    else:
        field_gap, field_tolerance, field_tied = within_one_std(field[first], field[second])
        if field_tied:
            value, clause = min((first, second), key=G2_METHODS.index), "within one std on mixed and on field -> simpler method"
        else:
            value, clause = max((first, second), key=lambda m: field[m].mean), "within one std -> higher val field macro-F1"
    return {"value": value, "decided_by": clause, "ranking": ranked, "top_two_gap": gap, "tolerance": tolerance,
            "field_gap": field_gap, "field_tolerance": field_tolerance}


def g2_candidate_experiments(choices, cut=()) -> dict:
    """{method: experiment ID} of the G2 candidates that are not cut. The plain one is the G1 backbone's unified row."""
    unknown = set(cut) - set(G2_METHOD_EXPERIMENTS)
    if unknown:
        raise ExperimentError(f"gate g2: only Method A or B can be cut, not {sorted(unknown)}")
    plain = G1_CANDIDATES[choices.get("backbone")]
    return {"plain": plain, **{m: exp_id for m, exp_id in G2_METHOD_EXPERIMENTS.items() if m not in cut}}


def evaluate_g2(exps: dict, cut=()) -> dict:
    """{"value": recommended experiment ID, "evidence"} from the complete candidates (keyed by method)."""
    mixed = {m: aggregate(exps[m], "mixed", "macro_f1", "val") for m in exps}
    field = {m: aggregate(exps[m], "field", "macro_f1", "val") for m in exps}
    decision = decide_g2(mixed, field)
    method = decision["value"]
    pairs = [(m, "plain") for m in ("A", "B") if m in exps] + ([("B", "A")] if "A" in exps and "B" in exps else [])
    as_float = lambda v: None if v is None else float(v)  # noqa: E731
    evidence = {
        "rule": G2_RULE, "metric": GATE_METRIC, "method": method, "method_label": G2_METHOD_LABELS[method],
        "decided_by": decision["decided_by"], "ranking": decision["ranking"],
        "top_two_gap": as_float(decision["top_two_gap"]), "tolerance_one_std": as_float(decision["tolerance"]),
        "field_gap": as_float(decision["field_gap"]), "field_tolerance_one_std": as_float(decision["field_tolerance"]),
        "cut": sorted(cut), "candidates": candidate_evidence(exps), "paired_differences": paired_evidence(exps, pairs),
    }
    return {"value": exps[method].exp_id, "evidence": evidence}


def describe_g2(result: dict) -> str:
    ev = result["evidence"]

    def cell(record: dict) -> str:
        return f"{record['mean']:.3f} +- {record['std']:.3f}"

    lines = [f"gate g2: {ev['metric']}", f"  {'candidate':<22}{'val mixed F1':>16}{'val field F1':>16}"]
    for method in ev["ranking"]:
        c = ev["candidates"][method]
        lines.append(f"  {method + ' (' + c['experiment'] + ')':<22}{cell(c['val_mixed_macro_f1']):>16}{cell(c['val_field_macro_f1']):>16}")
    for method in ev["cut"]:
        lines.append(f"  {method + ' (' + G2_METHOD_EXPERIMENTS[method] + ')':<22}{'cut':>16}{'cut':>16}")
    if ev["top_two_gap"] is not None:
        first, second = ev["ranking"][:2]
        within = "within" if ev["top_two_gap"] <= ev["tolerance_one_std"] else "outside"
        lines.append(f"  top two: {first} leads {second} by {ev['top_two_gap']:.3f} on mixed; one std = {ev['tolerance_one_std']:.3f} ({within} one std)")
        if ev["field_gap"] is not None:
            within = "within" if ev["field_gap"] <= ev["field_tolerance_one_std"] else "outside"
            lines.append(f"  on field they differ by {ev['field_gap']:.3f}; one std = {ev['field_tolerance_one_std']:.3f} ({within} one std)")
    lines.append(f"  decision: {result['value']} = {ev['method_label']}  ({ev['decided_by']})")
    return "\n".join(lines)


def run_gate_g2(data_root, choices, *, cut=(), force: bool = False, out=print) -> dict:
    """Gate G2 (see the section text): refuses incomplete candidates that are not cut, and a changed outcome once frozen."""
    names = g2_candidate_experiments(choices, cut)
    exps = {method: require_complete(data_root, exp_id, choices, "g2") for method, exp_id in names.items()}
    result = evaluate_g2(exps, cut)
    frozen = choices.raw("recommended")
    if frozen is not None and frozen != result["value"] and not force:
        out(describe_g2(result))
        raise ExperimentError(
            f"choices.json already recommends {frozen!r} (frozen by an earlier `gate g2`), and the runs now decide "
            f"{result['value']!r}. The recommendation is frozen once made (06: flipping it after seeing test numbers is "
            "test-set selection). If the earlier decision was a genuine mistake, rerun with --force and say so in the report"
        )
    path = write_choice(data_root, "recommended", result["value"], result["evidence"])
    out(describe_g2(result))
    out(f"  written to {path}: recommended = {result['value']!r}" + (" (overwrote a different frozen choice: --force)" if frozen not in (None, result["value"]) else ""))
    return result

### Phase 06 tables: `table_methods` and `table_segmenter`

* **`table_methods`**: plain (the G1 backbone's unified row) vs Method A (`E3-A`) vs Method B (`E4-B`), white / field /
  mixed / domain-balanced, plus the paired per-split difference of A and B minus plain. The **val** rows are the G2 evidence.
  The **test** rows stay `hidden until gate g2` until `choices.json["recommended"]` exists: 06 opens the test numbers of the
  methods only after the recommendation is frozen.
* **`table_segmenter`**: the five split-segmenters of `S-seg`, mean +- std (ddof=1) over the splits. IoU, Dice and the
  salient-leaf hit rate on the 20 **eval** annotations (all, and without the `ambiguous` ones) come from the `eval` section
  that `seg-eval --set eval --final` wrote into each `seg_metrics.json` (this table only reads it; while it is missing, or was
  measured on other boxes than the current ones, the rows read `pending`). Also: the mean crop-box area on those images
  (a box that always covers the whole image would always "hit"), fallbacks on the eval images and over all cached images
  per domain, and the validation IoUs that chose the checkpoint (white val pseudo-masks, val composites; raw mask at 0.5,
  before post-processing).

In [ ]:
#| export
METHOD_EXPERIMENTS = ("E3-A", "E4-B")
SEGMENTER_EXPERIMENT = "S-seg"
HIDDEN_UNTIL_G2 = "hidden until gate g2"


def plain_method_experiment(choices) -> str | None:
    """The G2 'plain' candidate: the unified row of the G1 backbone (None while gate G1 has not run)."""
    backbone = choices.raw("backbone")
    return G1_CANDIDATES.get(backbone) if backbone else None


def table_methods(results: dict, choices) -> Table:
    """Plain vs Method A vs Method B (G2 evidence on val; test only once the recommendation is frozen)."""
    plain = plain_method_experiment(choices)
    recommended = choices.raw("recommended")
    rows_spec = (("plain", plain), ("A", "E3-A"), ("B", "E4-B"))
    rows = []
    for part in PARTS:
        hidden = part == "test" and recommended is None
        for method, exp_id in rows_spec:
            label = exp_id or "(G1 backbone not chosen)"
            cells = [HIDDEN_UNTIL_G2] * len(SCORE_HEADER) if hidden else score_cells(results.get(exp_id) if exp_id else None, part)
            mark = " (recommended)" if recommended is not None and exp_id == recommended else ""
            rows.append([part, G2_METHOD_LABELS[method] + mark, label, *cells])
        for method, exp_id in rows_spec[1:]:
            cells = ([HIDDEN_UNTIL_G2] * len(SCORE_HEADER) if hidden
                     else difference_cells(results.get(exp_id), results.get(plain) if plain else None, part))
            rows.append([part, f"{method} - plain (paired)", f"{exp_id} - {plain or 'plain'}", *cells])
    return Table(
        "table_methods",
        "Methods: plain unified model vs Method A (composites) vs Method B (segment, crop)",
        ("Part", "Method", "Experiment", *SCORE_HEADER),
        rows,
        "Mean +- std over the 5 splits (std with ddof=1). Plain is the unified model of the G1 backbone; A adds composites "
        "to its training data; B crops every image to the box of its split's leaf segmenter. Gate G2 uses only the val rows. "
        "The paired rows are the per-split difference minus plain. Test rows are shown only after gate G2 froze the "
        "recommendation. Rows that are not complete yet read 'pending'.",
    )


def segmenter_eval_sections(exp: ExperimentResults | None, data_root) -> dict | None:
    """{split: eval section of seg_metrics.json} when every split has one measured on its current boxes, else None."""
    if exp is None:
        return None
    sections = {}
    for split_seed in exp.splits:
        section = read_seg_metrics(split_dir(runs_dir(data_root) / exp.run_name, split_seed)).get("eval")
        if not section or section.get("boxes_sha256") != exp.metrics[split_seed].get("boxes_sha256"):
            return None
        sections[split_seed] = section
    return sections


def annotation_counts(data_root, which: str) -> dict:
    """{"all": n, "excluding_ambiguous": n} of an annotation set (flags only, no measurement); '-' when unavailable."""
    try:
        entries = annotation_entries(data_root, which)
    except (ExperimentError, DatasetError):
        return {"all": "-", "excluding_ambiguous": "-"}
    return {"all": len(entries), "excluding_ambiguous": sum(not e["ambiguous"] for e in entries)}


def format_rate(stat) -> str:
    """'85.0 +- 5.0' (percent, 1 decimal) or 'n/a'."""
    return "n/a" if stat.mean != stat.mean or stat.std != stat.std else f"{100 * stat.mean:.1f} +- {100 * stat.std:.1f}"


def table_segmenter(exp: ExperimentResults | None, data_root) -> Table:
    """The leaf segmenter over its 5 splits (see the section text): eval annotations, fallbacks and the validation IoUs."""
    eval_sections = segmenter_eval_sections(exp, data_root)
    measure = "seg-eval --set eval --final"
    rows = []

    def add(label, images, values, fmt, *, needs_eval=False):
        if exp is None:
            rows.append([label, images, PENDING, PENDING])
        elif needs_eval and eval_sections is None:
            rows.append([label, images, f"{PENDING} ({measure})", PENDING])
        else:
            values = values()
            mean, std = mean_std(values)
            stat = Aggregate(mean, std, values)
            per_split = "; ".join("n/a" if v is None else (f"{100 * v:.1f}" if fmt is format_rate else f"{v:.3f}") for v in values)
            rows.append([label, images, fmt(stat) if fmt is format_rate else format_mean_std(stat, "macro_f1"), per_split])

    splits = exp.splits if exp is not None else ()
    n_eval = ({block: eval_sections[splits[0]]["summary"][block]["n"] for block in ("all", "excluding_ambiguous")} if eval_sections
              else annotation_counts(data_root, "eval"))
    for block, label in (("all", "eval annotations"), ("excluding_ambiguous", "eval annotations without the ambiguous ones")):
        n = n_eval[block]
        for key, name, fmt in (("iou", "IoU", None), ("dice", "Dice", None), ("hit_rate", "Salient-leaf hit rate (%)", format_rate),
                               ("box_area", "Crop box area (% of the image)", format_rate)):
            add(f"{name}, {label}", n, lambda b=block, k=key: [eval_sections[s]["summary"][b].get(k) for s in splits], fmt, needs_eval=True)
    add("Fallbacks on the eval annotations (%)", n_eval["all"], lambda: [eval_sections[s]["summary"]["fallbacks"] / max(1, eval_sections[s]["summary"]["all"]["n"]) for s in splits],
        format_rate, needs_eval=True)
    for domain in DOMAINS:
        n = exp.metrics[splits[0]]["fallbacks"][domain]["n"] if exp is not None else "-"
        add(f"Fallbacks, all cached {domain} images (%)", n, lambda d=domain: [exp.metrics[s]["fallbacks"][d]["rate"] for s in splits], format_rate)
    add("IoU, white val pseudo-masks (checkpoint data)", exp.metrics[splits[0]]["n_val_white"] if exp is not None else "-",
        lambda: [exp.metrics[s]["val_iou_white"] for s in splits], None)
    add("IoU, val composites (checkpoint data)", exp.metrics[splits[0]]["n_val_composites"] if exp is not None else "-",
        lambda: [exp.metrics[s]["val_iou_composite"] for s in splits], None)
    return Table(
        "table_segmenter",
        "Salient-leaf segmenter (S-seg, one per split)",
        ("Metric", "Images per split", "Mean +- std", "Per split (0-4)"),
        rows,
        "Mean +- std over the 5 split-segmenters (std with ddof=1). Eval annotations: the 20 hand-annotated field images that "
        "are looked at once per final segmenter. IoU and Dice compare the post-processed mask (the selected component; a "
        "fallback scores 0) with the annotated polygon at 512x256. Hit: the crop box (with the 15% margin, at the classifier's "
        "input aspect) contains at least 50% of the annotated leaf area; a fallback box is the whole image. Fallback: no "
        "component covers 2% of the image, so the whole image is used. The validation IoUs threshold the raw mask at 0.5.",
    )

## Phase 07 tables: per-class, hyperparameters, runtime (T7.2)

* **`table_per_class`**: the recommended method (`choices.json["recommended"]`), per class and domain: support (test images of the
  class summed over the 5 splits), precision, recall and F1 as mean +- std over the 5 test splits, plus the macro-F1 row.
* **`table_hparams`**: one row per reported experiment, every value read from the `config.json` of its five runs (never typed
  in). A value that differs between the splits of one experiment would read `varies: a / b` and be named in the caption. What the
  code fixes for all runs (optimiser, schedule, augmentation) is stated in the caption from the code's own constants.
* **`table_runtime`**: training time per run and seconds per epoch (mean +- std from `metrics.json`), epochs run, parameters, and
  the hardware of each experiment from `env.json`. Inference ms/image comes from `inference_benchmark.json`, which only the
  `benchmark` command writes (below); `summarize` never times anything, so its output stays reproducible. Without that file the
  column reads `not measured (run benchmark)`.
* `table_required` marks the recommended row like `table_methods` does, and adds it as a row if it is not one of the three plain models.

In [ ]:
#| export
BACKBONE_LABELS = {"resnet50": "ResNet50", "convnext_tiny": "ConvNeXt-T", "swin_t": "Swin-T"}
# Every run folder the report draws on, in table order (E1-U and E1-WW are aliases that read another run folder).
REPORTED_EXPERIMENTS = (*ANALYSIS_EXPERIMENTS, "E3-A", "E3-A-WF")


def recommended_experiment(choices) -> str | None:
    return choices.raw("recommended")


def per_class_stat(exp: ExperimentResults, part: str, subset: str, cls: str, key: str) -> Aggregate:
    """Precision, recall or F1 of one class on one subset: mean, std (ddof=1) and the per-split list."""
    values = [exp.metrics[k][part][subset]["per_class"][cls][key] for k in exp.splits]
    mean, std = mean_std(values)
    return Aggregate(mean, std, values)


def per_class_support(exp: ExperimentResults, part: str, subset: str, cls: str) -> int:
    return sum(exp.metrics[k][part][subset]["per_class"][cls]["support"] for k in exp.splits)


def summed_confusion(exp: ExperimentResults, subset: str, part: str = "test") -> np.ndarray:
    """The confusion matrices of the splits added up: rows = true class, columns = predicted class (CLASS_NAMES order)."""
    return sum(np.asarray(exp.metrics[k][part][subset]["confusion_matrix"], dtype=np.int64) for k in exp.splits)


def table_per_class(results: dict, choices) -> Table:
    """Per-class precision, recall and F1 of the recommended method on the white and the field test images."""
    exp_id = recommended_experiment(choices)
    exp = results.get(exp_id) if exp_id else None
    rows = []
    for domain in DOMAINS:
        for cls in CLASS_NAMES:
            if exp is None:
                rows.append([domain, cls, PENDING, PENDING, PENDING, PENDING])
                continue
            stats = [format_mean_std(per_class_stat(exp, "test", domain, cls, key), "macro_f1") for key in ("precision", "recall", "f1")]
            rows.append([domain, cls, per_class_support(exp, "test", domain, cls), *stats])
        if exp is None:
            rows.append([domain, "macro-F1", PENDING, "-", "-", PENDING])
        else:
            total = sum(per_class_support(exp, "test", domain, cls) for cls in CLASS_NAMES)
            rows.append([domain, "macro-F1", total, "-", "-", format_mean_std(aggregate(exp, domain, "macro_f1", "test"), "macro_f1")])
    return Table(
        "table_per_class",
        f"Per-class results of the recommended method ({exp_id or 'not chosen yet, gate g2'}), test split",
        ("Domain", "Class", "Support (5 splits)", "Precision", "Recall", "F1"),
        rows,
        "Mean +- std over the 5 test splits (std with ddof=1). Support is the number of test images of the class summed over the "
        "5 splits (an image can be in the test part of several splits). The macro-F1 row is the mean of the per-split macro-F1, "
        "not the mean of the class rows above it. A class that is never predicted in a split has precision 0 in that split.",
    )


def read_run_json(data_root, exp: ExperimentResults, split_seed: int, name: str) -> dict | None:
    """config.json or env.json of one split of an experiment's run folder (None when the file is missing)."""
    path = split_dir(runs_dir(data_root) / exp.run_name, split_seed) / name
    if not path.is_file():
        return None
    return json.loads(read_file_bytes(path).decode("utf-8"))


def hparam_cells(config: dict) -> dict:
    """The hyperparameters of one run as the table shows them, all read from its config.json."""
    e = config["experiment"]
    factor = config.get("choices_used", {}).get("composite_n_factor")
    return {
        "Backbone": e["backbone"],
        "Pretrained weights": config.get("weights") or "none",
        "Input mode": e["input_mode"],
        "Input HxW": "x".join(str(v) for v in config["input_size"]),
        "Train on": " + ".join(e["train_domains"]) + (" + composites" if e["use_composites"] else ""),
        "Checkpoint chosen on (val)": " + ".join(e["select_domains"]),
        "Composites per epoch": f"{factor:g}x" if factor is not None and e["use_composites"] else "-",
        "Linear-probe epochs x lr": f"{e['stage1_epochs']} x {e['stage1_lr']:g}",
        "Fine-tune epochs (max)": e["epochs"],
        "Learning rate": f"{e['lr']:g}",
        "Weight decay": f"{e['weight_decay']:g}",
        "Warmup epochs": e["warmup_epochs"],
        "Label smoothing": f"{e['label_smoothing']:g}",
        "Batch size x accumulation": f"{e['batch_size']} x {e['grad_accum']}",
        "Gradient clip norm": f"{e['clip_norm']:g}",
        "Early-stop patience": e["patience"],
        "Dropout": f"{e['dropout']:g}",
    }


HPARAM_COLUMNS = (
    "Backbone", "Pretrained weights", "Input mode", "Input HxW", "Train on", "Checkpoint chosen on (val)", "Composites per epoch",
    "Linear-probe epochs x lr", "Fine-tune epochs (max)", "Learning rate", "Weight decay", "Warmup epochs", "Label smoothing",
    "Batch size x accumulation", "Gradient clip norm", "Early-stop patience", "Dropout",
)


def alias_note(exp: ExperimentResults) -> str:
    """'' for a run folder that carries its own name, else 'E0-rect' (the run an alias such as E1-U reads)."""
    return "" if exp.run_name == exp.exp_id else exp.run_name


def table_hparams(results: dict, data_root) -> Table:
    """One row per reported experiment, every value read from the config.json of its five runs. A value that differs
    between the splits of one experiment is shown as 'varies: a / b' and named in the note."""
    rows, varying, missing = [], [], []
    for exp_id in REPORTED_EXPERIMENTS:
        exp = results.get(exp_id)
        if exp is None:
            rows.append([exp_id, "-", *[PENDING] * len(HPARAM_COLUMNS)])
            continue
        configs = [read_run_json(data_root, exp, k, "config.json") for k in exp.splits]
        if any(config is None for config in configs):
            missing.append(exp_id)
            rows.append([exp_id, exp.run_name, *["n/a"] * len(HPARAM_COLUMNS)])
            continue
        per_split = [hparam_cells(config) for config in configs]
        cells = []
        for column in HPARAM_COLUMNS:
            values = list(dict.fromkeys(str(cell[column]) for cell in per_split))
            if len(values) > 1:
                varying.append(f"{exp_id} ({column})")
            cells.append(values[0] if len(values) == 1 else "varies: " + " / ".join(values))
        rows.append([exp_id, exp.run_name, *cells])
    note = (
        "Every value is read from config.json of the experiment's runs (not typed in). "
        + ("Values are identical across the 5 splits of every experiment. " if not varying
           else "Not identical across the splits: " + ", ".join(varying) + ". ")
        + ("config.json is missing for: " + ", ".join(missing) + ". " if missing else "")
        + "E1-U and E1-WW are aliases that read the run folder named in the second column. Not part of config.json because they "
        "are fixed in the code: optimiser AdamW; stage 1 trains the classifier head only at a constant lr, stage 2 all weights with a linear warmup "
        "then cosine decay per optimiser step; mixed precision (fp16 autocast) on CUDA only, cross-entropy with the label smoothing "
        "above, checkpoint = best validation macro-F1 on the domains the model trained on. Training augmentation: random crop of "
        f"{AUG_CROP_AREA[0]:.0%}-{AUG_CROP_AREA[1]:.0%} of the area, random horizontal and vertical flips, rotation up to "
        f"+-{AUG_ROTATION_DEG:g} degrees, brightness +-{AUG_BRIGHTNESS:g}, contrast +-{AUG_CONTRAST:g}, saturation "
        f"+-{AUG_SATURATION:g}, hue +-{AUG_HUE:g}; evaluation images are only resized. Rows that are not complete yet read 'pending'."
    )
    return Table("table_hparams", "Hyperparameters of every reported experiment", ("Experiment", "Run folder", *HPARAM_COLUMNS), rows, note)

### Inference benchmark (`benchmark`)

`benchmark [--device cpu|mps|cuda]` builds the three backbones with untrained weights (the speed does not depend on the weights),
feeds a random image at the chosen input mode, and times the forward pass at batch size 1: 10 warm-up passes, then the median
(and quartiles) of 50 timed passes, with the device synchronised around every pass. It measures the model only (no image decoding
or resizing). The record of the device (platform, cores, GPU, torch version, precision, input size) is merged into
`_work/results/inference_benchmark.json` under its device name, so running it once per device collects them all. Timings depend on
the machine, so this file is committed as a measurement, not regenerated by `summarize`, which only reads it.

In [ ]:
#| export
BENCHMARK_FILENAME = "inference_benchmark.json"
BENCHMARK_FORMAT = 1
BENCHMARK_WARMUP = 10  # forward passes that are run and not timed
BENCHMARK_REPEATS = 50  # timed forward passes; the table reports their median
DEVICE_ORDER = ("cpu", "mps", "cuda")


def benchmark_path(data_root) -> Path:
    return results_dir(data_root) / BENCHMARK_FILENAME


def read_benchmark(data_root) -> dict:
    """{device: record} of inference_benchmark.json. {} when the file is missing (no benchmark run yet) or unusable."""
    path = benchmark_path(data_root)
    if not path.is_file():
        return {}
    try:
        data = json.loads(read_file_bytes(path).decode("utf-8"))
        devices = data["devices"]
        if data["format"] != BENCHMARK_FORMAT or not all(isinstance(r["backbones"], dict) for r in devices.values()):
            raise ValueError(f"unexpected format {data['format']!r}")
    except (ValueError, KeyError, TypeError, AttributeError) as exc:
        warn(f"{path}: unusable ({exc}); run `benchmark` again. Inference times read 'not measured'")
        return {}
    return {device: devices[device] for device in DEVICE_ORDER if device in devices}


def synchronize(device: str) -> None:
    if device == "cuda":
        torch.cuda.synchronize()
    elif device == "mps":
        torch.mps.synchronize()


def time_inference(model: nn.Module, device: str, size: tuple, *, warmup: int = BENCHMARK_WARMUP, repeats: int = BENCHMARK_REPEATS) -> dict:
    """Milliseconds of one forward pass on one image (batch 1): median and quartiles of ``repeats`` timed passes after
    ``warmup`` untimed ones. The device is synchronised around every pass, so asynchronous GPU work is counted."""
    model.eval()
    x = torch.rand(1, 3, *size, device=device)
    amp = device == "cuda"  # the same autocast the evaluation uses
    times = []
    with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
        for i in range(warmup + repeats):
            synchronize(device)
            started = time.perf_counter()
            model(x)
            synchronize(device)
            if i >= warmup:
                times.append((time.perf_counter() - started) * 1000.0)
    q1, median, q3 = (float(v) for v in np.percentile(times, [25, 50, 75]))
    return {"median_ms": round(median, 2), "q1_ms": round(q1, 2), "q3_ms": round(q3, 2)}


def run_benchmark(data_root, device: str, out=print, *, warmup: int = BENCHMARK_WARMUP, repeats: int = BENCHMARK_REPEATS) -> dict:
    """Time the three backbones at the chosen input mode on ``device`` (untrained weights: the speed does not depend on
    them) and merge the record into inference_benchmark.json, keeping the records of the other devices."""
    data_root = Path(data_root)
    mode = load_choices(data_root).get("input_mode")
    size = input_size(mode)
    env = environment_record(device)
    backbones = {}
    for name in BACKBONES:
        torch.manual_seed(0)
        model = build_model(name, pretrained=False).to(device)
        backbones[name] = {**time_inference(model, device, size, warmup=warmup, repeats=repeats), "parameters": count_parameters(model)}
        out(f"benchmark: {name} on {device}: {backbones[name]['median_ms']:.1f} ms/image (median of {repeats})")
        del model
        free_device_memory(device)
    record = {
        "device": device, "precision": precision_of(device), "input_mode": mode, "input_size": list(size), "batch_size": 1,
        "warmup": warmup, "repeats": repeats, "torch_threads": torch.get_num_threads(),
        **{key: env[key] for key in ("python", "platform", "torch", "torchvision", "cpu_count", "gpu", "cuda")},
        "measured": "model forward pass only (no image decoding or resizing), untrained weights, batch size 1",
        "backbones": backbones,
    }
    devices = read_benchmark(data_root)
    devices[device] = record
    path = benchmark_path(data_root)
    write_json(path, {"format": BENCHMARK_FORMAT, "devices": {d: devices[d] for d in DEVICE_ORDER if d in devices}}, indent=2)
    out(f"benchmark: wrote {path}")
    return record


def precision_of(device: str) -> str:
    """The numeric precision a run on this device uses: training and evaluation autocast to fp16 on CUDA only."""
    return "mixed precision (fp16 autocast)" if device == "cuda" else "fp32"


def host_label(env: dict) -> str:
    """Where a run was made, from its env.json: Kaggle and Colab leave their paths in torch_home."""
    home = env.get("torch_home") or ""
    if home.startswith("/kaggle"):
        return "Kaggle"
    if home.startswith("/content"):
        return "Colab"
    return "local " + str(env.get("platform", "")).split("-")[0]


def hardware_signature(exp: ExperimentResults | None, data_root) -> tuple | None:
    """The distinct (host, device, gpu, torch, python) of an experiment's five runs, read from env.json."""
    if exp is None:
        return None
    envs = [read_run_json(data_root, exp, k, "env.json") for k in exp.splits]
    if any(env is None for env in envs):
        return None
    return tuple(sorted({(host_label(env), env["device"], env.get("gpu") or "-", env["torch"], env["python"]) for env in envs}))


def hardware_text(signature: tuple | None, *, full: bool = True) -> str:
    """'Kaggle, cuda (Tesla T4), torch 2.11.0+cu128, Python 3.13.15, mixed precision (fp16 autocast)'; ``full=False`` stops at the device."""
    if not signature:
        return "n/a"
    parts = []
    for host, device, gpu, torch_version, python in signature:
        text = f"{host}, {device} ({gpu})"
        parts.append(text + (f", torch {torch_version}, Python {python}, {precision_of(device)}" if full else ""))
    return " / ".join(parts)


def format_number(stat, decimals: int = 1, scale: float = 1.0) -> str:
    """'8.5 +- 1.2' for a mean +- std of a plain quantity (minutes, seconds, epochs)."""
    if stat.mean != stat.mean or stat.std != stat.std:
        return "n/a"
    return f"{stat.mean * scale:.{decimals}f} +- {stat.std * scale:.{decimals}f}"


def metric_stat(exp: ExperimentResults, key: str) -> Aggregate:
    values = [exp.metrics[k][key] for k in exp.splits]
    mean, std = mean_std(values)
    return Aggregate(mean, std, values)


def table_runtime(results: dict, data_root) -> Table:
    """Training cost per experiment from metrics.json and env.json, and inference ms/image from the benchmark file."""
    benchmark = read_benchmark(data_root)
    devices = list(benchmark)
    header = ("Experiment", "Run folder", "Hardware", "Training time per run (min)", "Seconds per epoch", "Epochs run", "Parameters (M)",
              *([f"Inference ms/image, median (quartiles), {d}" for d in devices] or ["Inference ms/image"]))
    rows = []
    for exp_id in REPORTED_EXPERIMENTS:
        exp = results.get(exp_id)
        backbone = exp.config.backbone if exp is not None else EXPERIMENTS[exp_id].backbone
        if devices:
            timing = []
            for device in devices:
                entry = benchmark[device]["backbones"].get(backbone) if backbone else None
                timing.append("n/a" if entry is None else f"{entry['median_ms']:.1f} ({entry['q1_ms']:.1f}-{entry['q3_ms']:.1f})")
        else:
            timing = ["not measured (run benchmark)"]
        if exp is None:
            rows.append([exp_id, "-", *[PENDING] * (len(header) - 2)])
            continue
        hardware = hardware_signature(exp, data_root)
        rows.append([
            exp_id, exp.run_name, hardware_text(hardware, full=False),
            format_number(metric_stat(exp, "train_time_s"), 1, 1 / 60), format_number(metric_stat(exp, "seconds_per_epoch"), 1),
            format_number(metric_stat(exp, "epochs_run"), 1), f"{exp.metrics[exp.splits[0]]['parameters'] / 1e6:.1f}", *timing,
        ])
    where = "; ".join(
        f"{d}: {r['platform']}, {r['cpu_count']} cores, {r['gpu'] or 'no GPU'}, torch {r['torch']}, {r['input_mode']} input "
        f"{r['input_size'][0]}x{r['input_size'][1]}, {r['precision']}, median of {r['repeats']} after {r['warmup']} warm-up passes"
        for d, r in benchmark.items()
    )
    note = (
        "Training time and seconds per epoch are the mean +- std (ddof=1) over the 5 runs of an experiment; a run's time includes "
        "validation after every epoch, and the time is wall clock on a machine whose load varied (shared Kaggle/Colab sessions, "
        "a laptop), so it is only comparable within one hardware group and only roughly. Early stopping makes the epochs run "
        "differ between runs. Hardware is read from env.json (the findings list the groups with their torch versions). "
        "E1-U and E1-WW are aliases that read the run folder named in the second column. Parameters are those of the classifier "
        "with its 5-class head. Inference time is not measured by `summarize` (it would not be reproducible): the `benchmark` "
        "command writes inference_benchmark.json and this table reads it. It times the model forward pass for one image "
        "(batch 1), not image decoding, resizing or the first call. "
        + (f"Measured on - {where}." if benchmark else "No benchmark file yet, so inference times read 'not measured (run benchmark)'.")
    )
    return Table("table_runtime", "Training and inference cost", header, rows, note)

## Figures (T7.3)

All figures are PNG at 300 dpi, matplotlib only, sized for half an A4 column (4.4 inch wide or less), in the colour-blind-safe
Okabe-Ito palette. Classes always appear in `CLASS_NAMES` order; white is blue with circles, field orange with squares, mixed grey
with triangles, in every figure that separates them.

* `confusion_<recommended>_{white,field}.png`: the 5 test confusion matrices added up, colour = share of the true class (rows
  sum to 1), the number in a cell is the image count; the frame carries the domain colour.
* `confusion_<exp>_field.png` for `E1-WF`, `E2-swin-WF`, `E3-A-WF`: which classes collapse when the model never saw field images
  (phase-04 question 4), with E1-WF as the reference.
* `scenario_bars.png`: test macro-F1 for the white, field and mixed scenario of E1-U, E2-cnx-U, E2-swin-U and E3-A.
* `domain_gap.png`: field test macro-F1 by training regime (F->F, W->F, W + composites->F, U, A).
* `training_curves_<recommended>.png`: training loss and validation macro-F1 per epoch, the 5 splits overlaid; the grey band is the
  linear-probe stage and the star the epoch whose checkpoint was kept.

Bars show the mean over the 5 splits, whiskers the sample std (ddof=1), dots the single splits.

In [ ]:
#| export
# One marker per subset in every figure that shows scores (colours: DOMAIN_COLORS; mixed has no domain, so it is grey).
SUBSET_COLORS = {**DOMAIN_COLORS, "mixed": "#8C8C8C"}
SUBSET_MARKERS = {"white": "o", "field": "s", "mixed": "^"}
SPLIT_COLORS = (OKABE_ITO["black"], OKABE_ITO["sky"], OKABE_ITO["green"], OKABE_ITO["vermilion"], OKABE_ITO["purple"])  # one per split
SCENARIO_MODELS = ("E1-U", "E2-cnx-U", "E2-swin-U", "E3-A")  # the main models of scenario_bars.png
# domain_gap.png: (training regime, experiment). The regimes follow the phase-04 questions.
DOMAIN_GAP_ROWS = (
    ("F->F", "E1-FF"), ("W->F", "E1-WF"), ("W->F", "E2-cnx-WF"), ("W->F", "E2-swin-WF"), ("W + composites->F*", "E3-A-WF"),
    ("U (white + field)", "E1-U"), ("U (white + field)", "E2-swin-U"), ("A (U + composites)", "E3-A"),
)
WF_CONFUSION_EXPERIMENTS = ("E1-WF", "E2-swin-WF", "E3-A-WF")  # which classes collapse under W->F, with E1-WF as the reference
FIGURE_FONT = 7
WRAPPED_CLASSES = [name.replace(" ", "\n") for name in CLASS_NAMES]


def tidy_axes(ax) -> None:
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.set_axisbelow(True)


def finish_figure(fig, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=FIGURE_DPI)
    pyplot().close(fig)
    return path


def experiment_label(exp: ExperimentResults) -> str:
    return f"{exp.exp_id} ({BACKBONE_LABELS.get(exp.config.backbone, exp.config.backbone)})"


def save_confusion(data_root, exp: ExperimentResults, subset: str, path: Path | None = None) -> Path:
    """`confusion_<exp>_<subset>.png`: the test confusion matrices of the 5 splits added up; the colour is the share of the
    true class (each row sums to 1), the number in a cell is the image count. The frame has the domain's colour."""
    plt = pyplot()
    path = path or figures_dir(data_root) / f"confusion_{exp.exp_id}_{subset}.png"
    matrix = summed_confusion(exp, subset)
    share = matrix / np.maximum(matrix.sum(axis=1, keepdims=True), 1)
    trained = " + ".join(exp.config.train_domains)
    with plt.rc_context({"font.size": FIGURE_FONT}):
        fig, ax = plt.subplots(figsize=(3.9, 3.75))
        image = ax.imshow(share, cmap="Blues", vmin=0.0, vmax=1.0)
        for i in range(NUM_CLASSES):
            for j in range(NUM_CLASSES):
                ax.text(j, i, str(matrix[i, j]), ha="center", va="center", fontsize=FIGURE_FONT + 0.5,
                        fontweight="bold" if i == j else "normal", color="white" if share[i, j] > 0.55 else "black")
        ax.set_xticks(range(NUM_CLASSES), WRAPPED_CLASSES)
        ax.set_yticks(range(NUM_CLASSES), WRAPPED_CLASSES)
        ax.set_xlabel("Predicted class")
        ax.set_ylabel("True class")
        ax.set_title(f"{exp.exp_id}: trained on {trained}, tested on {subset} images\n"
                     f"(5 test splits added up, {int(matrix.sum())} predictions)", fontsize=FIGURE_FONT + 0.5)
        for spine in ax.spines.values():
            spine.set_edgecolor(DOMAIN_COLORS[subset])
            spine.set_linewidth(2.0)
        colorbar = fig.colorbar(image, ax=ax, fraction=0.046, pad=0.03)
        colorbar.set_label("Share of the true class")
        colorbar.outline.set_visible(False)
        fig.tight_layout(pad=0.6)
        return finish_figure(fig, path)


def save_scenario_bars(data_root, results: dict, recommended: str | None, path: Path | None = None) -> Path | None:
    """`scenario_bars.png`: test macro-F1 of the main models for the three scenarios (white, field, mixed); bars = mean over
    the 5 splits, whiskers = std (ddof=1), dots = the single splits."""
    plt = pyplot()
    path = path or figures_dir(data_root) / "scenario_bars.png"
    models = [m for m in SCENARIO_MODELS if m in results]
    if not models:
        return None
    subsets = ("white", "field", "mixed")
    width = 0.26
    with plt.rc_context({"font.size": FIGURE_FONT}):
        fig, ax = plt.subplots(figsize=(4.4, 3.0))
        for k, subset in enumerate(subsets):
            for position, model in enumerate(models):
                stat = aggregate(results[model], subset, "macro_f1", "test")
                x = position + (k - 1) * width
                ax.bar(x, stat.mean, width * 0.92, color=SUBSET_COLORS[subset], label=SUBSET_LABELS[subset] if position == 0 else None)
                ax.errorbar(x, stat.mean, yerr=stat.std, color="black", capsize=2, linewidth=0.8, capthick=0.8)
                jitter = np.linspace(-0.07, 0.07, len(stat.values))
                ax.scatter(x + jitter, stat.values, s=5, marker=SUBSET_MARKERS[subset], color="white", edgecolor="black", linewidth=0.4, zorder=3)
        ax.set_xticks(range(len(models)), [f"{m}\n{BACKBONE_LABELS.get(results[m].config.backbone, '')}"
                                           + ("\n(recommended)" if m == recommended else "") for m in models])
        ax.set_ylim(0, 1.0)
        ax.set_ylabel("Test macro-F1 (mean ± std, 5 splits)")
        ax.yaxis.grid(True, color="#DDDDDD", linewidth=0.6)
        tidy_axes(ax)
        ax.legend(frameon=False, ncol=3, loc="lower center", bbox_to_anchor=(0.5, 1.0), handlelength=1.2, columnspacing=1.2)
        fig.tight_layout(pad=0.6)
        return finish_figure(fig, path)


def save_domain_gap(data_root, results: dict, path: Path | None = None) -> Path | None:
    """`domain_gap.png`: test macro-F1 on the field images for each training regime (the transfer story). Bars = mean over the
    5 splits, whiskers = std (ddof=1), dots = the single splits."""
    plt = pyplot()
    path = path or figures_dir(data_root) / "domain_gap.png"
    rows = [(regime, exp_id) for regime, exp_id in DOMAIN_GAP_ROWS if exp_id in results]
    if not rows:
        return None
    ypos, last, y = [], None, 0.0
    for regime, _ in rows:
        y += 0.0 if last in (None, regime) else 0.6
        ypos.append(y)
        y += 1.0
        last = regime
    with plt.rc_context({"font.size": FIGURE_FONT}):
        fig, ax = plt.subplots(figsize=(4.4, 0.27 * (ypos[-1] + 1) + 1.0))
        for y, (regime, exp_id) in zip(ypos, rows):
            stat = aggregate(results[exp_id], "field", "macro_f1", "test")
            ax.barh(y, stat.mean, 0.72, color=DOMAIN_COLORS["field"])
            ax.errorbar(stat.mean, y, xerr=stat.std, color="black", capsize=2, linewidth=0.8, capthick=0.8)
            ax.scatter(stat.values, y + np.linspace(-0.2, 0.2, len(stat.values)), s=5, marker=SUBSET_MARKERS["field"], color="white",
                       edgecolor="black", linewidth=0.4, zorder=3)
            ax.text(min(1.0, max(stat.values + [stat.mean + stat.std])) + 0.015, y, f"{stat.mean:.3f}", va="center", fontsize=FIGURE_FONT)
        ax.set_yticks(ypos, [f"{regime}\n{exp_id}, {BACKBONE_LABELS.get(results[exp_id].config.backbone, '')}" for regime, exp_id in rows],
                      fontsize=FIGURE_FONT - 0.5)
        ax.set_ylim(ypos[-1] + 0.7, -0.7)
        ax.set_xlim(0, 1.0)
        ax.set_xlabel("Field test macro-F1 (mean ± std, 5 splits)")
        ax.xaxis.grid(True, color="#DDDDDD", linewidth=0.6)
        tidy_axes(ax)
        if any(exp_id == "E3-A-WF" for _, exp_id in rows):
            fig.text(0.01, 0.005, "* white photos only as leaves, but field images are the composite backgrounds: not pure W->F",
                     fontsize=FIGURE_FONT - 1.5, va="bottom", color="#444444")
        fig.tight_layout(pad=0.6, rect=(0, 0.03, 1, 1))
        return finish_figure(fig, path)


def read_history(data_root, exp: ExperimentResults, split_seed: int) -> list | None:
    """history.csv of one run as dicts (numbers as floats, stage as text); None when the file is missing."""
    path = split_dir(runs_dir(data_root) / exp.run_name, split_seed) / "history.csv"
    if not path.is_file():
        return None
    with open(path, newline="") as handle:
        return [{key: (value if key == "stage" else float(value)) for key, value in row.items()} for row in csv.DictReader(handle)]


def save_training_curves(data_root, exp: ExperimentResults, path: Path | None = None) -> Path | None:
    """`training_curves_<exp>.png`: training loss and validation macro-F1 per epoch, the 5 splits overlaid. The grey band is the
    linear-probe stage, the star the epoch whose checkpoint was kept."""
    plt = pyplot()
    path = path or figures_dir(data_root) / f"training_curves_{exp.exp_id}.png"
    histories = {k: read_history(data_root, exp, k) for k in exp.splits}
    histories = {k: h for k, h in histories.items() if h}
    if not histories:
        return None
    selected = " + ".join(exp.config.select_domains)
    with plt.rc_context({"font.size": FIGURE_FONT}):
        fig, (loss_ax, f1_ax) = plt.subplots(2, 1, figsize=(4.4, 4.2), sharex=True)
        for k, history in histories.items():
            color = SPLIT_COLORS[k % len(SPLIT_COLORS)]
            epochs = [row["epoch"] for row in history]
            loss_ax.plot(epochs, [row["train_loss"] for row in history], color=color, linewidth=1.0, label=f"split {k}")
            f1_ax.plot(epochs, [row["val_macro_f1"] for row in history], color=color, linewidth=1.0)
            kept = [row for row in history if row["best"]][-1:]
            f1_ax.scatter([row["epoch"] for row in kept], [row["val_macro_f1"] for row in kept], marker="*", s=40, color=color,
                          edgecolor="black", linewidth=0.5, zorder=3)
        probe = max((row["epoch"] for h in histories.values() for row in h if row["stage"] == "lp"), default=0)
        for ax in (loss_ax, f1_ax):
            if probe:
                ax.axvspan(0.5, probe + 0.5, color="#E6E6E6", zorder=0)
            ax.yaxis.grid(True, color="#DDDDDD", linewidth=0.6)
            tidy_axes(ax)
        if probe:
            loss_ax.text(probe / 2 + 0.5, 0.4, "linear\nprobe", transform=loss_ax.get_xaxis_transform(), ha="center", va="top",
                         fontsize=FIGURE_FONT - 1)
        loss_ax.set_ylabel("Training loss")
        f1_ax.set_ylabel(f"Validation macro-F1\n({selected})")
        f1_ax.set_xlabel("Epoch (star = kept checkpoint)")
        f1_ax.set_ylim(0, 1.0)
        loss_ax.set_title(f"{exp.exp_id}: training curves of the 5 splits", fontsize=FIGURE_FONT + 0.5)
        loss_ax.legend(frameon=False, ncol=5, loc="upper right", fontsize=FIGURE_FONT - 1, handlelength=1.2, columnspacing=0.8)
        fig.tight_layout(pad=0.6)
        return finish_figure(fig, path)


def write_figures(data_root, results: dict, choices) -> list:
    """The T7.3 figures that the available runs support. Returns the paths written."""
    recommended = recommended_experiment(choices)
    paths = []
    if recommended in results:
        paths += [save_confusion(data_root, results[recommended], subset) for subset in DOMAINS]
        paths.append(save_training_curves(data_root, results[recommended]))
    for exp_id in WF_CONFUSION_EXPERIMENTS:
        if exp_id in results:
            paths.append(save_confusion(data_root, results[exp_id], "field"))
    paths += [save_scenario_bars(data_root, results, recommended), save_domain_gap(data_root, results)]
    return [p for p in paths if p is not None]

## Success, failure and challenging cases (T7.4)

Every image sits in the test part of a different number of splits (about 0.75 of them on average), so the test predictions of
the recommended method are pooled over the 5 splits and each image is judged by all the splits that tested it:

* **consistent failure**: wrong in every split that tested it;
* **consistent success**: right in every split that tested it, and its top probability never below 0.5;
* **challenging**: everything else: right in some splits and wrong in others, or right but with a top probability below 0.5 in some split.

`cases.csv` lists every image with its category, the predicted class per split (blank where it was not tested), the modal
prediction and the mean probability of the true and of the modal class. The contact sheets show 10 images each, picked
deterministically and balanced: one image per (class, domain) group in turn, white and field alternating, so no class or domain
with cases is skipped before another is repeated. Within a group successes are the most confident, failures the most often tested
(then the most confidently wrong), challenging cases the ones whose outcomes split closest to half. The label reads
`true -> predicted (mean probability of the predicted class)` and the filename with how often the image was right.
The photos come from the image cache, so the sheets are skipped (with a warning) where it is missing.

In [ ]:
#| export
CASE_LOW_CONFIDENCE = 0.5  # a correct prediction whose top probability is below this is not a success but a challenging case
CASE_SHEET_IMAGES = 10  # images per contact sheet: 2 rows of 5
CASE_CATEGORIES = ("success", "failure", "challenging")
CASE_SHEET_COLUMNS = 5
CASES_FILENAME = "cases.csv"


def classify_case(correct: list, confidence: list) -> str:
    """One image's outcomes over the splits whose test part held it: ``correct[i]`` and the top probability
    ``confidence[i]`` of each. failure = wrong every time; success = right every time and never below
    CASE_LOW_CONFIDENCE; challenging = everything else (mixed outcomes, or right but unsure)."""
    if not any(correct):
        return "failure"
    if all(correct) and min(confidence) >= CASE_LOW_CONFIDENCE:
        return "success"
    return "challenging"


def pool_predictions(data_root, exp: ExperimentResults) -> dict:
    """{filename: {"domain", "true", "tests": {split: (predicted class index, probabilities)}}} from predictions_test.csv of
    every split of ``exp``. An image is in the test part of about 0.75 splits on average, so most have one to three entries."""
    pooled = {}
    for split_seed in exp.splits:
        path = split_dir(runs_dir(data_root) / exp.run_name, split_seed) / "predictions_test.csv"
        if not path.is_file():
            raise ExperimentError(f"{exp.exp_id}: {path} is missing")
        with open(path, newline="") as handle:
            for row in csv.DictReader(handle):
                probs = [float(row[f"p{i}"]) for i in range(NUM_CLASSES)]
                entry = pooled.setdefault(row["filename"], {"domain": row["domain"], "true": row["true"], "tests": {}})
                entry["tests"][split_seed] = (CLASS_NAMES.index(row["pred"]), probs)
    return pooled


def case_rows(pooled: dict) -> list:
    """One dict per image (sorted by filename): outcome counts, category, the split-wise predictions, the modal prediction
    (ties go to the higher summed probability) and the probabilities of the true and the modal class, averaged over the splits."""
    rows = []
    for filename in sorted(pooled):
        entry = pooled[filename]
        true_index = CLASS_NAMES.index(entry["true"])
        tests = entry["tests"]
        correct = [pred == true_index for pred, _ in tests.values()]
        top = [max(probs) for _, probs in tests.values()]
        counts = Counter(pred for pred, _ in tests.values())
        wrong = [pred for pred, _ in counts.most_common() if pred != true_index]  # wrong classes, the most frequent first
        modal = max(counts, key=lambda c: (counts[c], sum(probs[c] for _, probs in tests.values()), -c))
        rows.append({
            "filename": filename, "domain": entry["domain"], "true": entry["true"], "tested": len(tests), "correct": sum(correct),
            "category": classify_case(correct, top),
            "preds": [CLASS_NAMES[tests[k][0]] if k in tests else "" for k in ANALYSIS_SPLITS],
            "modal_pred": CLASS_NAMES[modal], "wrong_pred": CLASS_NAMES[wrong[0]] if wrong else "",
            "mean_p_modal": float(np.mean([probs[modal] for _, probs in tests.values()])),
            "mean_p_true": float(np.mean([probs[true_index] for _, probs in tests.values()])),
            "min_top_p": float(min(top)),
        })
    return rows


def cases_csv(rows: list) -> str:
    header = ["filename", "domain", "true", "tested", "correct", "category", *[f"pred_split{k}" for k in ANALYSIS_SPLITS],
              "modal_pred", "wrong_pred", "mean_p_modal", "mean_p_true", "min_top_p"]
    return csv_text(header, [[r["filename"], r["domain"], r["true"], r["tested"], r["correct"], r["category"], *r["preds"], r["modal_pred"],
                              r["wrong_pred"], f"{r['mean_p_modal']:.3f}", f"{r['mean_p_true']:.3f}", f"{r['min_top_p']:.3f}"] for r in rows])


# Which images a contact sheet shows first within one (domain, class): successes the model is surest of, failures that were tested
# most often (the most evidence) and are the most confidently wrong, challenging cases that split the splits closest to half.
CASE_ORDER = {
    "success": lambda r: (-r["mean_p_modal"], -r["tested"], r["filename"]),
    "failure": lambda r: (-r["tested"], -r["mean_p_modal"], r["filename"]),
    "challenging": lambda r: (-r["tested"], abs(r["correct"] / r["tested"] - 0.5), r["filename"]),
}


def select_cases(rows: list, category: str, n: int = CASE_SHEET_IMAGES) -> list:
    """Up to ``n`` images of a category for its contact sheet, deterministic and balanced: one image per (class, domain) group in
    turn, white and field alternating, until ``n`` are picked, so a domain or class with cases is shown before any is repeated.
    The result is ordered white then field, by class."""
    queues = []
    for cls in CLASS_NAMES:
        for domain in DOMAINS:
            group = [r for r in rows if r["category"] == category and r["domain"] == domain and r["true"] == cls]
            if group:
                queues.append(sorted(group, key=CASE_ORDER[category]))
    picked = []
    while len(picked) < n and any(queues):
        for queue in queues:
            if queue and len(picked) < n:
                picked.append(queue.pop(0))
    return sorted(picked, key=lambda r: (DOMAINS.index(r["domain"]), CLASS_NAMES.index(r["true"]), r["filename"]))


def case_label(row: dict) -> str:
    """'true ->' over 'predicted (p)' over 'filename  correct/tested'; p is the mean probability of the predicted (modal) class.
    A fourth line names the wrong class an image that is mostly right was also taken for."""
    text = f"{row['true']} ->\n{row['modal_pred']} ({row['mean_p_modal']:.2f})\n{row['filename']}  {row['correct']}/{row['tested']} correct"
    if row["wrong_pred"] and row["modal_pred"] == row["true"]:
        text += f"\nalso taken for {row['wrong_pred']}"
    return text


def save_case_sheet(data_root, rows: list, category: str, exp_id: str, path: Path | None = None) -> Path:
    """A contact sheet `cases_<category>.png` (2 rows of 5): the photo (domain-coloured frame and tag), the label
    `true -> predicted (p)` and how often the image was right. The photos are drawn 1:1 at the figure's resolution."""
    plt = pyplot()
    path = path or figures_dir(data_root) / f"cases_{category}.png"
    picks = select_cases(rows, category)
    n_rows = max(1, math.ceil(len(picks) / CASE_SHEET_COLUMNS))
    width_in, margin, gap, tile_w, tile_h, label_h, head = 5.8, 0.08, 0.05, 1.08, 1.9, 0.46, 0.22
    pitch = (width_in - 2 * margin - tile_w) / (CASE_SHEET_COLUMNS - 1)  # left edge to left edge of neighbouring tiles
    height_in = head + n_rows * (tile_h + label_h + gap) + margin
    adjective = {"success": "consistent successes", "failure": "consistent failures", "challenging": "challenging cases"}[category]
    with plt.rc_context({"font.size": 6}):
        fig = plt.figure(figsize=(width_in, height_in))
        fig.text(margin / width_in, 1 - 0.04 / height_in, f"{exp_id}: {adjective} (test images pooled over the 5 splits; label: true -> predicted (mean p))",
                 fontsize=6.3, fontweight="bold", va="top")
        if not picks:
            fig.text(0.5, 0.5, f"no {adjective} in the test predictions", ha="center", va="center", fontsize=8)
        for slot, row in enumerate(picks):
            r, c = divmod(slot, CASE_SHEET_COLUMNS)
            cell_left = margin + c * pitch
            cell_top = height_in - head - r * (tile_h + label_h + gap)
            rgb = read_cached_rgb(data_root, row["filename"])
            scale = min(tile_w * FIGURE_DPI / rgb.shape[1], tile_h * FIGURE_DPI / rgb.shape[0])
            px = (max(1, round(rgb.shape[1] * scale)), max(1, round(rgb.shape[0] * scale)))
            w, h = px[0] / FIGURE_DPI, px[1] / FIGURE_DPI
            ax = fig.add_axes(((cell_left + (tile_w - w) / 2) / width_in, (cell_top - tile_h + (tile_h - h) / 2) / height_in, w / width_in, h / height_in))
            ax.imshow(cv2.resize(rgb, px, interpolation=cv2.INTER_AREA), interpolation="none", aspect="auto")
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_edgecolor(DOMAIN_COLORS[row["domain"]])
                spine.set_linewidth(1.6)
            ax.text(0.04, 0.985, DOMAIN_LABELS[row["domain"]].split()[0], transform=ax.transAxes, ha="left", va="top", fontsize=5.5,
                    fontweight="bold", color="white" if row["domain"] == "white" else "black",
                    bbox={"boxstyle": "square,pad=0.2", "fc": DOMAIN_COLORS[row["domain"]], "ec": "none"})
            fig.text((cell_left + tile_w / 2) / width_in, (cell_top - tile_h - 0.03) / height_in, case_label(row), ha="center", va="top",
                     fontsize=5.6, linespacing=1.15)
        return finish_figure(fig, path)


def write_cases(data_root, results: dict, choices, out=print) -> list | None:
    """T7.4: cases.csv and the three contact sheets for the recommended method. None while it is not chosen or not complete."""
    exp_id = recommended_experiment(choices)
    exp = results.get(exp_id) if exp_id else None
    if exp is None:
        warn("cases: skipped, the recommended method is not chosen (gate g2) or not complete")
        return None
    rows = case_rows(pool_predictions(data_root, exp))
    path = results_dir(data_root) / CASES_FILENAME
    write_text_atomic(path, cases_csv(rows))
    out(f"summarize: wrote {path}")
    shown = [name for category in CASE_CATEGORIES for name in (r["filename"] for r in select_cases(rows, category))]
    if not all((cache_dir(data_root) / name).is_file() for name in shown):
        warn("cases: contact sheets skipped, the image cache is incomplete (run `prepare`)")
        return rows
    for category in CASE_CATEGORIES:
        out(f"summarize: wrote {save_case_sheet(data_root, rows, category, exp.exp_id)}")
    return rows

### Generated findings: domain questions, cases, hardware

Three more auto sections of `findings.md`, all numbers read from the runs: the four phase-04 questions (domain gap, does white
help field, does field hurt white, which classes collapse under W->F), the case counts of T7.4, and the hardware of each run
folder with the comparisons that cross hardware groups (`env.json` records device, GPU and torch; nothing is typed in).

In [ ]:
#| export
def have(results: dict, *exp_ids) -> bool:
    return all(exp_id in results for exp_id in exp_ids)


def pending_sentence(*exp_ids) -> str:
    return "Not available yet: " + ", ".join(exp_ids) + " is not complete."


def f1_text(exp: ExperimentResults, subset: str = "field", part: str = "test") -> str:
    return format_mean_std(aggregate(exp, subset, "macro_f1", part), "macro_f1")


def paired_text(a: ExperimentResults, b: ExperimentResults, subset: str = "field", part: str = "test") -> str:
    """'+0.012 +- 0.034; E2-swin-U ahead in 3 of 5 splits' for the per-split difference a - b."""
    diff = paired_difference(a, b, subset, "macro_f1", part)
    return f"{diff.mean:+.3f} +- {diff.std:.3f}; {a.exp_id} ahead in {diff.ahead} of {len(diff.values)} splits"


def paired_mean(a: ExperimentResults, b: ExperimentResults, subset: str = "field", part: str = "test") -> float:
    return paired_difference(a, b, subset, "macro_f1", part).mean


def direction(value: float, up: str, down: str) -> str:
    return up if value > 0 else down


def per_class_f1_text(exp: ExperimentResults, subset: str = "field") -> str:
    return ", ".join(f"{cls} {per_class_stat(exp, 'test', subset, cls, 'f1').mean:.3f}" for cls in CLASS_NAMES)


def largest_confusions(exp: ExperimentResults, subset: str = "field", top: int = 2) -> str:
    """The most frequent mistakes (true class predicted as another class) of the summed test confusion matrix, with the share of
    the true class's images they make up."""
    matrix = summed_confusion(exp, subset)
    mistakes = sorted(((int(matrix[i, j]), i, j) for i in range(NUM_CLASSES) for j in range(NUM_CLASSES) if i != j and matrix[i, j] > 0),
                      key=lambda m: (-m[0], m[1], m[2]))[:top]
    return "; ".join(f"{CLASS_NAMES[i]} taken for {CLASS_NAMES[j]} in {n} of {int(matrix[i].sum())} images ({n / matrix[i].sum():.0%})"
                     for n, i, j in mistakes) or "no mistakes"


COLLAPSE_RATIO = 0.5  # a class has collapsed under W->F when its field F1 is below this share of the field-trained model's (E1-FF)


def domain_questions_findings(results: dict) -> str:
    """Answers to the four phase-04 questions, in numbers read from the test metrics (field/white macro-F1, mean +- std over the
    5 splits; paired = per-split difference)."""
    paragraphs = []
    if have(results, "E1-FF", "E1-WF"):
        ff, wf = results["E1-FF"], results["E1-WF"]
        gap = paired_mean(wf, ff)
        text = (f"**1. Domain gap.** Field test macro-F1 is {f1_text(ff)} for E1-FF (ResNet50 trained on field images) and {f1_text(wf)} for "
                f"E1-WF (trained on white images only), so training on white alone {direction(gap, 'raises', 'lowers')} field macro-F1 by "
                f"{abs(gap):.3f} (paired E1-WF - E1-FF: {paired_text(wf, ff)}).")
        for exp_id in ("E2-cnx-WF", "E2-swin-WF"):
            if exp_id in results:
                text += f" {exp_id} (white only, {BACKBONE_LABELS[results[exp_id].config.backbone]}) reaches {f1_text(results[exp_id])}."
        if "E3-A-WF" in results:
            extra = f" (paired E3-A-WF - E2-swin-WF: {paired_text(results['E3-A-WF'], results['E2-swin-WF'])})" if "E2-swin-WF" in results else ""
            text += (f" With composites E3-A-WF reaches {f1_text(results['E3-A-WF'])}{extra}; its training leaves are white photos, but the "
                     "backgrounds are field images, so it is not pure W->F.")
        paragraphs.append(text)
    else:
        paragraphs.append("**1. Domain gap.** " + pending_sentence("E1-FF", "E1-WF"))
    if have(results, "E1-U", "E1-FF"):
        u, ff = results["E1-U"], results["E1-FF"]
        diff = paired_mean(u, ff)
        paragraphs.append(f"**2. Does white data help field?** Field test macro-F1 is {f1_text(u)} for E1-U (white + field) and {f1_text(ff)} for E1-FF "
                          f"(field only): the unified model is {direction(diff, 'ahead', 'behind')} by {abs(diff):.3f} (paired E1-U - E1-FF: {paired_text(u, ff)}).")
    else:
        paragraphs.append("**2. Does white data help field?** " + pending_sentence("E1-U", "E1-FF"))
    if have(results, "E1-U", "E1-WW"):
        u, ww = results["E1-U"], results["E1-WW"]
        diff = paired_mean(u, ww, "white")
        paragraphs.append(f"**3. Does field data hurt white?** White test macro-F1 is {f1_text(u, 'white')} for E1-U (white + field) and {f1_text(ww, 'white')} for "
                          f"E1-WW (white only, the E1-WF run): the unified model is {direction(diff, 'ahead', 'behind')} by {abs(diff):.3f} "
                          f"(paired E1-U - E1-WW: {paired_text(u, ww, 'white')}).")
    else:
        paragraphs.append("**3. Does field data hurt white?** " + pending_sentence("E1-U", "E1-WW"))
    rows = [exp_id for exp_id in ("E1-WF", "E2-cnx-WF", "E2-swin-WF", "E3-A-WF") if exp_id in results]
    if rows and "E1-FF" in results:
        lines = [f"- E1-FF (field only, reference): per-class field F1 {per_class_f1_text(results['E1-FF'])}."]
        for exp_id in rows:
            exp = results[exp_id]
            change = {cls: per_class_stat(exp, "test", "field", cls, "f1").mean - per_class_stat(results["E1-FF"], "test", "field", cls, "f1").mean
                      for cls in CLASS_NAMES}
            collapsed = [cls for cls in CLASS_NAMES
                         if per_class_stat(exp, "test", "field", cls, "f1").mean < COLLAPSE_RATIO * per_class_stat(results["E1-FF"], "test", "field", cls, "f1").mean]
            worst = min(change, key=lambda cls: (change[cls], CLASS_NAMES.index(cls)))
            lines.append(f"- {exp_id}: per-class field F1 {per_class_f1_text(exp)}. Collapsed: {', '.join(collapsed) if collapsed else 'none'}. "
                         f"Largest change against E1-FF: {worst} ({change[worst]:+.3f}). Most frequent mistakes: {largest_confusions(exp)}.")
        paragraphs.append("**4. Which classes collapse under W->F?** Per-class field test F1 (mean over the 5 splits). A class counts as collapsed when its "
                          f"F1 is below {COLLAPSE_RATIO:.0%} of E1-FF's for that class. The mistakes are counted in the "
                          "confusion matrices added up over the 5 test splits (`confusion_<exp>_field.png`).\n\n" + "\n".join(lines))
    else:
        paragraphs.append("**4. Which classes collapse under W->F?** " + pending_sentence("E1-FF", "E1-WF"))
    return "\n\n".join(paragraphs)


HARDWARE_COMPARISONS = (("E1-FF", "E1-WF"), ("E1-U", "E1-FF"), ("E1-U", "E1-WW"), ("E2-cnx-U", "E1-U"), ("E2-swin-U", "E1-U"),
                        ("E2-swin-U", "E2-cnx-U"), ("E2-swin-WF", "E1-WF"), ("E3-A", "E2-swin-U"), ("E3-A-WF", "E2-swin-WF"))


def hardware_findings(results: dict, data_root) -> str:
    """Which runs were made on which hardware (from env.json), and which comparisons of the tables cross hardware. '' when no
    complete experiment has an env.json."""
    readers = {}  # run folder -> the experiment IDs that read it (E1-U reads E0-rect)
    for exp_id in REPORTED_EXPERIMENTS:
        if exp_id in results:
            readers.setdefault(results[exp_id].run_name, []).append(exp_id)
    groups = {}
    for run_name, exp_ids in readers.items():
        signature = hardware_signature(results[exp_ids[0]], data_root)
        aliases = [e for e in exp_ids if e != run_name]
        if signature:
            groups.setdefault(signature, []).append(run_name + (f" (also {', '.join(aliases)})" if aliases else ""))
    lines = [f"- {', '.join(names)}: {hardware_text(signature)}" for signature, names in groups.items()]
    if not lines:
        return ""
    text = "Hardware per run folder, read from each run's env.json:\n\n" + "\n".join(lines)
    if len(groups) > 1:
        crossing = []
        for a, b in HARDWARE_COMPARISONS:
            if have(results, a, b) and hardware_signature(results[a], data_root) != hardware_signature(results[b], data_root):
                crossing.append(f"{a} vs {b}")
        text += ("\n\nThe runs were not all made on the same hardware, torch build and precision, so a small difference between experiment "
                 "families can include hardware and numerics effects (fp16 autocast on CUDA against fp32 on the Mac, different GPUs and "
                 "torch builds), not only the method. Comparisons inside one group share their hardware. Comparisons made in the tables that "
                 "cross groups: " + (", ".join(crossing) if crossing else "none") + ". Read those gaps against the paired std. Training "
                 "times (table_runtime) are only comparable inside a group.")
    else:
        text += "\n\nAll runs share the same hardware, torch build and precision, so comparisons between experiments carry no hardware effect."
    return text


def cases_findings(rows: list, exp_id: str) -> str:
    """Counts of the T7.4 case categories for the recommended method, by domain, and where the failures concentrate."""
    total = len(rows)
    once = sum(r["tested"] == 1 for r in rows)
    by = lambda category, domain=None: [r for r in rows if r["category"] == category and (domain is None or r["domain"] == domain)]  # noqa: E731
    sentences = [f"Pooled test predictions of {exp_id} over the 5 splits cover {total} distinct images; {once} of them were in the test part of only one split."]
    names = {"success": "consistent successes (right in every split they were tested in, top probability never below "
                        f"{CASE_LOW_CONFIDENCE:g})", "failure": "consistent failures (wrong in every split they were tested in)",
             "challenging": f"challenging cases (right in some splits and wrong in others, or right but below {CASE_LOW_CONFIDENCE:g} top probability)"}
    for category in CASE_CATEGORIES:
        found = by(category)
        sentences.append(f"{len(found)} {names[category]}: {len(by(category, 'white'))} white, {len(by(category, 'field'))} field.")
    failures = by("failure")
    if failures:
        pairs = Counter((r["true"], r["modal_pred"]) for r in failures).most_common(3)
        sentences.append("Most frequent failure pairs (true -> predicted): " + "; ".join(f"{t} -> {p} ({n})" for (t, p), n in pairs) + ".")
        tested_once = sum(r["tested"] == 1 for r in failures)
        sentences.append(f"{tested_once} of the {len(failures)} failures were tested in one split only, so for them 'every time' rests on one prediction.")
    sentences.append("The per-image list is `cases.csv`; the contact sheets are `cases_success.png`, `cases_failure.png` and `cases_challenging.png`.")
    return " ".join(sentences)

## `summarize` and `findings.md`

`summarize` writes `table_input_mode`, `table_required`, `table_domains`, `table_methods`, `table_segmenter`, `table_per_class`,
`table_hparams` and `table_runtime`, the T7.3 figures, `cases.csv` with its three contact sheets, and the "G0 input mode", "G2
recommended method", "domain transfer questions", "cases" and "hardware" sections of `findings.md`. The generated sections sit between
`<!-- auto:NAME:begin/end -->` markers; text outside the markers (Nattan's failure-mode notes, for example) is preserved on every
run. Everything is computed from the run folders in well under two minutes on a CPU (no checkpoint is loaded).

In [ ]:
#| export
FINDINGS_TITLE = (
    "# Findings\n\n"
    "Sections between `<!-- auto:NAME:begin -->` and `<!-- auto:NAME:end -->` markers are regenerated by "
    "`summarize` from the run folders. Anything outside the markers is yours and is never touched.\n"
)


def auto_block(name: str, heading: str, body: str) -> str:
    return f"<!-- auto:{name}:begin -->\n## {heading}\n\n{body.strip()}\n<!-- auto:{name}:end -->"


def update_findings_text(existing: str | None, blocks: dict) -> str:
    """Replace each auto block in ``existing`` (append the ones that are not there yet); keep everything else."""
    text = existing if existing else FINDINGS_TITLE
    for name, block in blocks.items():
        pattern = re.compile(rf"<!-- auto:{re.escape(name)}:begin -->.*?<!-- auto:{re.escape(name)}:end -->", re.DOTALL)
        if pattern.search(text):
            text = pattern.sub(lambda _match: block, text, count=1)
        else:
            text = text.rstrip("\n") + "\n\n" + block + "\n"
    return text


def g0_findings(result: dict) -> str:
    """The G0 section of findings.md in plain English, with every number taken from the gate evidence."""
    ev, value = result["evidence"], result["value"]

    def f1(mode: str) -> str:
        record = ev["candidates"][mode]["val_mixed_macro_f1"]
        return f"{record['mean']:.3f} +- {record['std']:.3f}"

    height, width = INPUT_SIZES[value]
    order = ev["ranking"]
    sentences = [
        f"Gate G0 picked the {value} input ({height}x{width}) with the rule fixed in phase 04: highest mean validation mixed "
        "macro-F1, and if the top two are within one std, prefer rect.",
        "Validation mixed macro-F1 (mean +- std over 5 splits) was " + ", ".join(f"{m} {f1(m)}" for m in order[:-1])
        + f" and {order[-1]} {f1(order[-1])}.",
    ]
    leader = order[0]
    if ev["decided_by"].startswith("within one std") and leader != value:
        sentences.append(
            f"{leader.capitalize()} leads {value} by {ev['top_two_gap']:.3f}, less than one std ({ev['tolerance_one_std']:.3f}), so {value} was "
            "preferred because it keeps the leaf geometry at no extra cost."
        )
        paired = ev["paired_differences"][f"{leader}_minus_{value}"]
        mixed, field = paired["val_mixed_macro_f1"], paired["val_field_macro_f1"]
        sentences.append(
            f"The paired {leader}-minus-{value} validation difference is {mixed['mean']:+.3f} +- {mixed['std']:.3f} on mixed and "
            f"{field['mean']:+.3f} +- {field['std']:.3f} on field macro-F1, with {leader} ahead on field in "
            f"{field['splits_ahead']} of {len(field['per_split'])} splits; this is recorded as a finding, not as a reason to revisit the choice."
        )
    else:
        sentences.append(f"{value} has the highest mean, so the tie-break clause was not needed.")
    return " ".join(sentences)


def g2_findings(result: dict) -> str:
    """The G2 section of findings.md, every number taken from the frozen evidence in choices.json."""
    ev = result["evidence"]

    def f1(method: str, key: str) -> str:
        record = ev["candidates"][method][key]
        return f"{record['mean']:.3f} +- {record['std']:.3f}"

    named = lambda m: f"{G2_METHOD_LABELS[m]} ({ev['candidates'][m]['experiment']})"  # noqa: E731
    sentences = [
        f"Gate G2 recommends {'the ' if ev['method'] == 'plain' else ''}{named(ev['method'])}, by the rule fixed in phase 06: highest mean validation mixed macro-F1; "
        "if the top two are within one std, the higher validation field macro-F1; if that is also within one std, the simpler method.",
        "Validation mixed / field macro-F1 (mean +- std over 5 splits): "
        + "; ".join(f"{named(m)} {f1(m, 'val_mixed_macro_f1')} / {f1(m, 'val_field_macro_f1')}" for m in ev["ranking"]) + ".",
    ]
    if ev["cut"]:
        sentences.append("Left out because of a cut line: " + ", ".join(f"Method {m}" for m in ev["cut"]) + ".")
    sentences.append(f"Decided by: {ev['decided_by']}.")
    for name, pair in ev["paired_differences"].items():
        a, b = name.split("_minus_")
        mixed, field = pair["val_mixed_macro_f1"], pair["val_field_macro_f1"]
        sentences.append(f"Paired {a} - {b} on validation: mixed {mixed['mean']:+.3f} +- {mixed['std']:.3f} "
                         f"({mixed['splits_ahead']} of {len(mixed['per_split'])} splits ahead), field {field['mean']:+.3f} +- {field['std']:.3f}.")
    return " ".join(sentences)


def summarize(data_root, out=print) -> None:
    """Regenerate every table, figure and findings section that the runs available so far support. Experiments that are
    incomplete or do not match the registry are skipped with a warning."""
    data_root = Path(data_root)
    choices = load_choices(data_root)
    results = load_experiments(data_root, (*ANALYSIS_EXPERIMENTS, *METHOD_EXPERIMENTS, SEGMENTER_EXPERIMENT), choices)
    optional = [exp_id for exp_id in OPTIONAL_DOMAIN_EXPERIMENTS if (runs_dir(data_root) / exp_id).is_dir()]
    results.update(load_experiments(data_root, optional, choices))
    tables = (table_input_mode(results), table_required(results, choices), table_domains(results, choices), table_methods(results, choices),
              table_segmenter(results.get(SEGMENTER_EXPERIMENT), data_root), table_per_class(results, choices),
              table_hparams(results, data_root), table_runtime(results, data_root))
    for table in tables:
        for path in write_table(data_root, table):
            out(f"summarize: wrote {path}")
    blocks = {}
    e0 = {mode: results.get(f"E0-{mode}") for mode in INPUT_MODES}
    if all(e0.values()):
        g0 = evaluate_g0(e0)
        recorded = choices.raw("input_mode")
        if recorded != g0["value"]:
            warn(f"choices.json has input_mode = {recorded!r} but the E0 runs decide {g0['value']!r}; run `gate g0`")
        blocks["g0"] = auto_block("g0", "G0 input mode", g0_findings(g0))
    else:
        warn("findings.md: G0 section skipped, the three E0 experiments are not all complete")
    recommended = choices.data.get("recommended")
    if isinstance(recommended, dict) and recommended.get("evidence"):
        blocks["g2"] = auto_block("g2", "G2 recommended method", g2_findings(recommended))
    for path in write_figures(data_root, results, choices):
        out(f"summarize: wrote {path}")
    cases = write_cases(data_root, results, choices, out)
    if any(exp_id in results for exp_id in ("E1-U", "E1-FF", "E1-WF")):
        blocks["questions"] = auto_block("questions", "Domain transfer questions (phase 04)", domain_questions_findings(results))
    if cases is not None:
        blocks["cases"] = auto_block("cases", "Success, failure and challenging cases (T7.4)", cases_findings(cases, recommended_experiment(choices)))
    hardware = hardware_findings(results, data_root)
    if hardware:
        blocks["hardware"] = auto_block("hardware", "Hardware", hardware)
    path = results_dir(data_root) / "findings.md"
    existing = read_file_bytes(path).decode("utf-8") if path.is_file() else None
    write_text_atomic(path, update_findings_text(existing, blocks))
    out(f"summarize: wrote {path}")

## Grad-CAM: the shortcut check (T7.5)

Does the model look at the leaf or at the background? `gradcam` explains the plain model (the recommended `E2-swin-U`) and Method A
(`E3-A`) with Grad-CAM (see `explain_images` in the classifier notebook) and answers it in two ways:

* **Quantitatively**, on the 30 hand-annotated field images (`annotations/field_salient_leaf.json`, dev and eval sets together:
  the dev images only tuned the segmenter, never a classifier). Each image is counted in every split where it was a **test or a
  validation** image and never where it was a training image, because a trained model would show the leaf it memorised there.
  That makes (image, split) *pairs*; one image can give up to two or three pairs. For each pair and model, the **energy
  inside** is the share of the Grad-CAM map (upsampled, not normalised, ReLU-ed) that falls inside the annotated leaf polygon;
  the **leaf area share** is the polygon's area over the image's: a uniform map would score exactly that, so energy above the
  area share means the model's evidence sits on the leaf more than chance. The map explains the class the model predicted
  (`energy_pred`, the headline) and, as a second view, the true class (`energy_true`). As for the segmenter, the numbers are given
  over all annotated images and without the ones flagged ambiguous.
* **Qualitatively**: `gradcam_success.png` and `gradcam_failure.png` show the images of the T7.4 contact sheets (the same
  selection, `select_cases`), with the photo, the plain model's map and Method A's map side by side, on one split where the
  image was a test image. `gradcam_annotated.png` shows annotated images with the polygon outline over both maps.

The checkpoints are the retrained copies in `runs/retrain/<run folder>/split_k/best.pt` (the reported runs keep none), unless
`--runs-dir` points elsewhere. Their predictions are the reported runs' (checked on every image explained: the predicted class
is compared with the reported `predictions_*.csv`). Pairs are not independent (an image can appear in two splits, and both models see
the same images), so the std is a spread, not a standard error, and 24 images are a small sample: read the paired difference as
a size and a direction, not as a test.

Outputs: `gradcam_energy.csv` (one row per pair), `table_gradcam.{csv,md}`, the three figures, and the `auto:gradcam` section
of `findings.md` (owned by `gradcam`; `summarize` leaves it alone and loads no checkpoint). With `--split K` only split K is used
and every file gets a `_splitK` suffix (no `findings.md` section).

In [ ]:
#| export
GRADCAM_COMPARE_EXPERIMENT = "E3-A"  # the second model: Method A (Method B was cut at gate G2)
GRADCAM_MODEL_NAMES = {"E2-swin-U": "Plain", "E3-A": "Method A"}
GRADCAM_PARTS = ("test", "val")  # the parts of a split in which an annotated image counts (never the training part)
GRADCAM_KEEP_PX = 480  # the maps kept for the figures are shrunk to this long side (the energy uses the full-size map)
GRADCAM_BATCH = 8
GRADCAM_ENERGY_FILENAME = "gradcam_energy.csv"
GRADCAM_TABLE_NAME = "table_gradcam"


def gradcam_label(exp_id: str) -> str:
    """'Plain (E2-swin-U)' for the known models, the ID otherwise."""
    return f"{GRADCAM_MODEL_NAMES[exp_id]} ({exp_id})" if exp_id in GRADCAM_MODEL_NAMES else exp_id


def checkpoint_run_dir(data_root, exp_id: str, choices, split_seed: int, runs_override=None) -> Path:
    """The split folder holding ``best.pt`` for Grad-CAM: <runs_override>/<run folder>/split_k when given, else the reported run
    folder if it kept a checkpoint, else runs/retrain/<run folder>/split_k (the retrained copies of batch5-checkpoints)."""
    folder = run_folder_name(exp_id, choices)
    roots = [Path(runs_override)] if runs_override is not None else [runs_dir(data_root), runs_dir(data_root) / RETRAIN_RUNS_FOLDER]
    candidates = [split_dir(root / folder, split_seed) for root in roots]
    for candidate in candidates:
        if (candidate / "best.pt").is_file():
            return candidate
    raise ExperimentError(
        f"{exp_id} split {split_seed}: no best.pt in {' or '.join(str(c) for c in candidates)}. The reported runs keep no checkpoints: "
        "fetch runs/retrain/<ID>/split_k/ (Kaggle batch `batch5-checkpoints`, docs/instructions/kaggle-guide.md) or pass --runs-dir DIR"
    )


def reported_predictions(data_root, exp_id: str, choices, split_seed: int, part: str) -> dict | None:
    """{filename: predicted class index} of the reported run (predictions_<part>.csv); None when the file is missing."""
    path = split_dir(runs_dir(data_root) / run_folder_name(exp_id, choices), split_seed) / f"predictions_{part}.csv"
    if not path.is_file():
        return None
    with open(path, newline="") as handle:
        return {row["filename"]: CLASS_NAMES.index(row["pred"]) for row in csv.DictReader(handle)}


def annotated_pairs(data_root, annotations=None, splits=ANALYSIS_SPLITS) -> list:
    """The (image, split) pairs of the shortcut check: every annotated field image (dev and eval sets) in every split where it was a
    test or a validation image, as the annotation entry plus ``split`` and ``part``. Sorted by image, then split."""
    entries = [entry for which in SEG_EVAL_SETS for entry in annotation_entries(data_root, which, annotations)]
    members = {k: {part: set(load_split(data_root, k)[part]) for part in GRADCAM_PARTS} for k in splits}
    pairs = [{**entry, "split": k, "part": part} for entry in entries for k in splits for part in GRADCAM_PARTS if entry["filename"] in members[k][part]]
    return sorted(pairs, key=lambda pair: (pair["filename"], pair["split"]))


@dataclass(frozen=True)
class Measurement:
    """What Grad-CAM said about one image under one model."""

    probs: np.ndarray  # [5]
    pred: int
    cam: np.ndarray  # the predicted class's map, not normalised, long side <= GRADCAM_KEEP_PX
    energy_pred: float | None  # share of the map inside the polygon (None without a polygon, NaN for an all-zero map)
    energy_true: float | None  # the same for the true class's map
    area: float | None  # the polygon's share of the image (the chance level of the energy share)
    reported: bool | None  # the reported run predicted the same class (None: no predictions file)


def cam_energy_share(cam: np.ndarray, mask: np.ndarray) -> float:
    """Share of the map's total energy (sum of the non-normalised map) inside ``mask``; NaN for an all-zero map."""
    total = float(cam.sum(dtype=np.float64))
    return float(cam[mask].sum(dtype=np.float64)) / total if total > 0.0 else float("nan")


def shrink_cam(cam: np.ndarray, long_side: int = GRADCAM_KEEP_PX) -> np.ndarray:
    scale = long_side / max(cam.shape)
    if scale >= 1.0:
        return cam.astype(np.float32)
    return cv2.resize(cam, (max(1, round(cam.shape[1] * scale)), max(1, round(cam.shape[0] * scale))), interpolation=cv2.INTER_AREA)


def measure_model(data_root, choices, exp_id: str, tasks: dict, device: str, runs_override=None, out=print) -> dict:
    """Grad-CAM of one model on ``tasks``, {(split, filename): {"true": class index, "part": "test"|"val", "polygon": fractional
    (x, y) array or None}}. Each split's checkpoint is loaded once. Returns {(split, filename): Measurement}; the energy shares are
    computed on the full-size maps, for the predicted class and for the true class, when the task has a polygon."""
    measured = {}
    for split_seed in sorted({k for k, _ in tasks}):
        model, checkpoint = load_run_model(checkpoint_run_dir(data_root, exp_id, choices, split_seed, runs_override), device)
        keys = sorted(key for key in tasks if key[0] == split_seed)
        reported = {part: reported_predictions(data_root, exp_id, choices, split_seed, part) for part in GRADCAM_PARTS}
        out(f"gradcam: {exp_id} split {split_seed}: {len(keys)} images on {device}")
        for start in range(0, len(keys), GRADCAM_BATCH):
            chunk = keys[start : start + GRADCAM_BATCH]
            rgbs = [read_cached_rgb(data_root, name) for _, name in chunk]
            predicted = explain_images(model, checkpoint, rgbs, device)
            other = [i for i, key in enumerate(chunk) if tasks[key]["polygon"] is not None and tasks[key]["true"] != predicted[i].predicted]
            truth = dict(zip(other, explain_images(model, checkpoint, [rgbs[i] for i in other], device, [tasks[chunk[i]]["true"] for i in other]))) if other else {}
            for i, key in enumerate(chunk):
                task, explanation = tasks[key], predicted[i]
                energy_pred = energy_true = area = None
                if task["polygon"] is not None:
                    mask = rasterize_polygon(task["polygon"], rgbs[i].shape[:2])
                    area = float(mask.mean())
                    energy_pred = cam_energy_share(explanation.cam, mask)
                    energy_true = energy_pred if i not in truth else cam_energy_share(truth[i].cam, mask)
                table = reported[task["part"]]
                measured[key] = Measurement(explanation.probs, explanation.predicted, shrink_cam(explanation.cam), energy_pred, energy_true, area,
                                            None if table is None else table.get(key[1]) == explanation.predicted)
        del model
    return measured


def energy_rows(pairs: list, measured: dict, exp_ids: tuple) -> list:
    """One dict per pair for the CSV and the summaries: the annotation fields, ``area`` (the polygon's share of the image, the
    chance level) and ``measured[exp_id]`` per model (a Measurement)."""
    rows = []
    for pair in pairs:
        key = (pair["split"], pair["filename"])
        rows.append({"filename": pair["filename"], "cls": pair["cls"], "set": pair["set"], "ambiguous": pair["ambiguous"],
                     "split": pair["split"], "part": pair["part"],
                     "area": measured[exp_ids[0]][key].area,
                     **{exp_id: measured[exp_id][key] for exp_id in exp_ids}})
    return rows


def energy_csv(rows: list, exp_ids: tuple) -> str:
    header = ["filename", "class", "set", "ambiguous", "split", "part", "leaf_area_share"]
    for exp_id in exp_ids:
        header += [f"{exp_id}_pred", f"{exp_id}_p_pred", f"{exp_id}_p_true", f"{exp_id}_energy_pred", f"{exp_id}_energy_true", f"{exp_id}_same_as_reported"]
    body = []
    for r in rows:
        line = [r["filename"], r["cls"], r["set"], str(r["ambiguous"]).lower(), r["split"], r["part"], f"{r['area']:.4f}"]
        for exp_id in exp_ids:
            m = r[exp_id]
            line += [CLASS_NAMES[m.pred], f"{m.probs[m.pred]:.3f}", f"{m.probs[CLASS_NAMES.index(r['cls'])]:.3f}", f"{m.energy_pred:.4f}",
                     f"{m.energy_true:.4f}", "" if m.reported is None else str(m.reported).lower()]
        body.append(line)
    return csv_text(header, body)


# The subsets of the pairs that the table and the findings report, as (label, predicate on a row).
ENERGY_SUBSETS = (
    ("all pairs", lambda r: True),
    ("test pairs", lambda r: r["part"] == "test"),
    ("val pairs", lambda r: r["part"] == "val"),
    ("all pairs, without the ambiguous images", lambda r: not r["ambiguous"]),
    ("test pairs, without the ambiguous images", lambda r: r["part"] == "test" and not r["ambiguous"]),
)
ENERGY_KEYS = (("energy_pred", "predicted class"), ("energy_true", "true class"))


def energy_stats(rows: list, plain: str, other: str, key: str) -> dict:
    """n (pairs with a finite value for both models), the images they cover, the mean leaf area share, mean and sample std
    (ddof=1) of the energy inside for both models and of the paired difference other - plain, and the pairs in which ``other``
    is ahead."""
    pairs = [(r["area"], getattr(r[plain], key), getattr(r[other], key), r["filename"]) for r in rows]
    pairs = [p for p in pairs if np.isfinite(p[1]) and np.isfinite(p[2])]
    area, base, new = (np.array([p[i] for p in pairs], dtype=np.float64) for i in range(3))
    return {"n": len(pairs), "images": len({p[3] for p in pairs}), "area": float(area.mean()) if len(pairs) else float("nan"),
            plain: mean_std(list(base)), other: mean_std(list(new)), "diff": mean_std(list(new - base)), "ahead": int((new > base).sum())}


def per_image_rows(rows: list, exp_ids: tuple) -> list:
    """One row per image: the pairs of an image averaged (its splits), so that every image counts once."""
    merged = []
    for filename in sorted({r["filename"] for r in rows}):
        group = [r for r in rows if r["filename"] == filename]
        row = {**group[0], "area": float(np.mean([r["area"] for r in group]))}
        for exp_id in exp_ids:
            row[exp_id] = Measurement(group[0][exp_id].probs, group[0][exp_id].pred, group[0][exp_id].cam,
                                      float(np.mean([r[exp_id].energy_pred for r in group])), float(np.mean([r[exp_id].energy_true for r in group])), row["area"], None)
        merged.append(row)
    return merged


def format_stat(stat: tuple) -> str:
    return f"{stat[0]:.3f} +- {stat[1]:.3f}"


def table_gradcam(rows: list, plain: str, other: str, name: str = GRADCAM_TABLE_NAME) -> Table:
    """The shortcut check as a table: for the predicted-class and the true-class maps, per subset of the pairs, the leaf area share
    (chance level), the energy inside the leaf polygon of both models (mean +- std over the pairs) and the paired difference."""
    table_rows = []
    for key, key_label in ENERGY_KEYS:
        blocks = [(label, [r for r in rows if keep(r)]) for label, keep in ENERGY_SUBSETS] + [("all pairs, each image once (its splits averaged)", per_image_rows(rows, (plain, other)))]
        for label, subset in blocks:
            if not subset:
                continue
            s = energy_stats(subset, plain, other, key)
            table_rows.append([key_label, label, f"{s['n']} ({s['images']} images)", f"{s['area']:.3f}", format_stat(s[plain]), format_stat(s[other]),
                               f"{s['diff'][0]:+.3f} +- {s['diff'][1]:.3f}", f"{s['ahead']} of {s['n']}"])
    return Table(
        name, f"Grad-CAM energy inside the annotated leaf: {gradcam_label(plain)} vs {gradcam_label(other)}",
        ("CAM of the", "Subset", "Pairs", "Leaf area share (chance)", f"{plain} energy inside", f"{other} energy inside", f"{other} - {plain} (paired)", f"{other} ahead in"),
        table_rows,
        "Pairs are (annotated field image, split) where the image was a test or validation image, never a training image. Energy inside = the "
        "share of the Grad-CAM map (upsampled, not normalised) inside the annotated salient-leaf polygon; the leaf area share is what a uniform "
        "map would score. Mean +- std (ddof=1) over the pairs; the paired difference is per pair, other minus plain. The pairs of one image are "
        "not independent, hence the last block of each part, where each image counts once.",
    )

### Grad-CAM figures

All three figures share one layout (`save_cam_sheet`, 5.8 inch wide, 300 dpi): groups of one photo and one map per model, two groups
per row, every tile in the frame colour of the image's domain. A map is scaled to [0, 1] on its own (so tiles show where a model
looks, not how strongly) and blended over the photo with the `turbo` colour map at 45%: red is the most evidence for the class,
blue the least. Under a map the label reads `true -> predicted (p)` (red if wrong) for that model on that split; under a photo the
file name and the split. The annotated-leaf figure also draws the polygon outline (black under yellow, as in
`annotations_overview.png`) and prints the energy inside under each map.

In [ ]:
#| export
import textwrap

GRADCAM_ALPHA = 0.45  # weight of the colour map in the blend with the photo
GRADCAM_COLORMAP = "turbo"
GRADCAM_ANNOTATED_IMAGES = 8
FOOTER_CHARS = 150  # characters per footer line at the footer's font size on a 5.8 inch sheet


def blend_cam(rgb: np.ndarray, cam: np.ndarray, px: tuple[int, int]) -> np.ndarray:
    """The photo shrunk to ``px`` = (width, height) with the normalised map in colour over it (weight GRADCAM_ALPHA). uint8 RGB."""
    photo = cv2.resize(rgb, px, interpolation=cv2.INTER_AREA).astype(np.float32)
    heat = cv2.resize(normalise_cam(cam).astype(np.float32), px, interpolation=cv2.INTER_LINEAR)
    colour = pyplot().get_cmap(GRADCAM_COLORMAP)(np.clip(heat, 0.0, 1.0))[..., :3] * 255.0
    return np.clip((1.0 - GRADCAM_ALPHA) * photo + GRADCAM_ALPHA * colour + 0.5, 0, 255).astype(np.uint8)


def save_cam_sheet(path: Path, items: list, headings: list, title: str, footer: str) -> Path:
    """A sheet of Grad-CAM groups. Each item: ``rgb``, ``domain``, ``caption`` (under the photo), ``cams`` (one map per heading
    after the first), ``labels`` (one text per map) and optionally ``wrong`` (a bool per map, label in red) and ``polygon``
    (fractional outline drawn on every tile). ``headings`` names the tiles of a group (the photo first)."""
    plt = pyplot()
    per_group, groups_per_row = len(headings), 2
    width_in, margin, group_gap, tile_gap, tile_h, label_h, row_gap, head = 5.8, 0.08, 0.16, 0.03, 1.45, 0.34, 0.07, 0.36
    tile_w = (width_in - 2 * margin - (groups_per_row - 1) * group_gap - groups_per_row * (per_group - 1) * tile_gap) / (groups_per_row * per_group)
    n_rows = max(1, math.ceil(len(items) / groups_per_row))
    footer = textwrap.fill(footer, width=FOOTER_CHARS)
    foot = 0.1 + 0.085 * (footer.count("\n") + 1)
    height_in = head + n_rows * (tile_h + label_h + row_gap) + foot
    with plt.rc_context({"font.size": 6}):
        fig = plt.figure(figsize=(width_in, height_in))
        fig.text(margin / width_in, 1 - 0.04 / height_in, title, fontsize=6.3, fontweight="bold", va="top")
        fig.text(margin / width_in, 0.05 / height_in, footer, fontsize=5.2, va="bottom", color="#333333", linespacing=1.2)
        for slot, item in enumerate(items):
            r, g = divmod(slot, groups_per_row)
            top = height_in - head - r * (tile_h + label_h + row_gap)
            rgb = item["rgb"]
            scale = min(tile_w * FIGURE_DPI / rgb.shape[1], tile_h * FIGURE_DPI / rgb.shape[0])
            px = (max(1, round(rgb.shape[1] * scale)), max(1, round(rgb.shape[0] * scale)))
            w, h = px[0] / FIGURE_DPI, px[1] / FIGURE_DPI
            for t in range(per_group):
                left = margin + g * (group_gap + per_group * tile_w + (per_group - 1) * tile_gap) + t * (tile_w + tile_gap)
                ax = fig.add_axes(((left + (tile_w - w) / 2) / width_in, (top - tile_h + (tile_h - h) / 2) / height_in, w / width_in, h / height_in))
                tile = cv2.resize(rgb, px, interpolation=cv2.INTER_AREA) if t == 0 else blend_cam(rgb, item["cams"][t - 1], px)
                ax.imshow(tile, extent=(0, 1, 1, 0), interpolation="none", aspect="auto")
                if item.get("polygon") is not None:
                    closed = np.vstack([item["polygon"], item["polygon"][:1]])
                    ax.plot(closed[:, 0], closed[:, 1], color="black", linewidth=1.2, solid_joinstyle="round")
                    ax.plot(closed[:, 0], closed[:, 1], color=ANNOTATION_OUTLINE, linewidth=0.6, solid_joinstyle="round")
                ax.set_xlim(0, 1)
                ax.set_ylim(1, 0)
                ax.set_xticks([])
                ax.set_yticks([])
                for spine in ax.spines.values():
                    spine.set_edgecolor(DOMAIN_COLORS[item["domain"]])
                    spine.set_linewidth(1.4)
                if t == 0:
                    ax.text(0.04, 0.985, DOMAIN_LABELS[item["domain"]].split()[0], transform=ax.transAxes, ha="left", va="top", fontsize=5.2,
                            fontweight="bold", color="white" if item["domain"] == "white" else "black",
                            bbox={"boxstyle": "square,pad=0.2", "fc": DOMAIN_COLORS[item["domain"]], "ec": "none"})
                if r == 0:
                    fig.text((left + tile_w / 2) / width_in, (top + 0.04) / height_in, headings[t], ha="center", va="bottom", fontsize=5.6, fontweight="bold")
                wrong = t > 0 and item.get("wrong", [False] * per_group)[t - 1]
                fig.text((left + tile_w / 2) / width_in, (top - tile_h - 0.03) / height_in, item["caption"] if t == 0 else item["labels"][t - 1],
                         ha="center", va="top", fontsize=5.2, linespacing=1.15, color=OKABE_ITO["vermilion"] if wrong else "black")
        return finish_figure(fig, path)


def cam_label(measure: Measurement, true_index: int, extra: str = "") -> str:
    """'true ->' over 'predicted (p)' (and an optional third line), for under a map."""
    return f"{CLASS_NAMES[true_index]} ->\n{CLASS_NAMES[measure.pred]} ({measure.probs[measure.pred]:.2f})" + (f"\n{extra}" if extra else "")


def case_items(data_root, picks: list, measured: dict, exp_ids: tuple) -> list:
    """Sheet items for ``picks`` = [(case row, split)]: the photo and one map per model, labelled on that split."""
    items = []
    for row, split_seed in picks:
        key, true_index = (split_seed, row["filename"]), CLASS_NAMES.index(row["true"])
        found = [measured[exp_id][key] for exp_id in exp_ids]
        items.append({"rgb": read_cached_rgb(data_root, row["filename"]), "domain": row["domain"], "caption": f"{row['filename']}\nsplit {split_seed}",
                      "cams": [m.cam for m in found], "labels": [cam_label(m, true_index) for m in found], "wrong": [m.pred != true_index for m in found]})
    return items


def select_annotated_examples(rows: list, n: int = GRADCAM_ANNOTATED_IMAGES) -> list:
    """The pairs of the annotated-leaf figure: test pairs of images that are not flagged ambiguous, one image per class in turn
    (images in filename order, each once, its first split), so no class is skipped before another is repeated. No look at the
    energies."""
    queues = []
    for cls in CLASS_NAMES:
        seen, queue = set(), []
        for r in sorted((r for r in rows if r["cls"] == cls and r["part"] == "test" and not r["ambiguous"]), key=lambda r: (r["filename"], r["split"])):
            if r["filename"] not in seen:
                seen.add(r["filename"])
                queue.append(r)
        queues.append(queue)
    picked = []
    while len(picked) < n and any(queues):
        for queue in queues:
            if queue and len(picked) < n:
                picked.append(queue.pop(0))
    return picked


def annotated_items(data_root, pairs_by_key: dict, rows: list, exp_ids: tuple) -> list:
    """Sheet items of the annotated-leaf figure: photo with the polygon, and each model's map with its energy inside."""
    items = []
    for r in rows:
        true_index = CLASS_NAMES.index(r["cls"])
        found = [r[exp_id] for exp_id in exp_ids]
        items.append({"rgb": read_cached_rgb(data_root, r["filename"]), "domain": "field", "polygon": pairs_by_key[(r["split"], r["filename"])]["polygon"],
                      "caption": f"{r['filename']}\nsplit {r['split']}, leaf {r['area']:.0%} of image",
                      "cams": [m.cam for m in found], "labels": [cam_label(m, true_index, f"inside {m.energy_pred:.0%}") for m in found],
                      "wrong": [m.pred != true_index for m in found]})
    return items

### `gradcam`: findings section and the command

`gradcam_findings` writes the `auto:gradcam` section of `findings.md` from the pair rows: the numbers first, then a plain reading. The
reading follows one fixed rule, stated in the text: the paired difference is judged on the per-image averages (24 independent
images, not 43 related pairs); it counts as a difference only when it is more than 2 standard errors from zero, and the sign must
be the same in the sub-groups for the reading to be called consistent. `run_gradcam` is the command: it explains the models on
the union of the figure images and the annotated pairs (each model's checkpoint for a split is loaded once), writes the files and
the section.

In [ ]:
#| export
GRADCAM_SIGNIFICANT_SE = 2.0  # a paired difference counts as a difference when it is this many standard errors from zero


def gradcam_reading(rows: list, plain: str, other: str) -> str:
    """The plain-language reading of the shortcut check, by the fixed rule: judge the difference on the per-image averages (every
    image once); it is a difference only beyond GRADCAM_SIGNIFICANT_SE standard errors of zero."""
    images = energy_stats(per_image_rows(rows, (plain, other)), plain, other, "energy_pred")
    mean, std = images["diff"]
    standard_error = std / math.sqrt(images["n"])
    sub_signs = {math.copysign(1.0, energy_stats(subset, plain, other, "energy_pred")["diff"][0])
                 for subset in ([r for r in rows if keep(r)] for _, keep in ENERGY_SUBSETS) if subset}
    consistent = len(sub_signs) == 1 and math.copysign(1.0, mean) in sub_signs
    name, base = gradcam_label(other), gradcam_label(plain)
    if abs(mean) < GRADCAM_SIGNIFICANT_SE * standard_error:
        verdict = (f"{name} and {base} put about the same share of their map on the leaf (per-image difference {mean:+.3f}, "
                   f"{abs(mean) / standard_error:.1f} standard errors from zero), so this check shows no effect of the composites on where the model looks.")
    elif mean > 0:
        verdict = (f"{name} puts more of its map on the annotated leaf than {base} (per-image difference {mean:+.3f}, {mean / standard_error:.1f} standard "
                   f"errors from zero), which is the direction the motivation of Method A predicts: composites put leaves on mismatched backgrounds, "
                   "so the background should carry less evidence.")
    else:
        verdict = (f"{name} puts less of its map on the annotated leaf than {base} (per-image difference {mean:+.3f}, {abs(mean) / standard_error:.1f} "
                   "standard errors from zero), the opposite of what the motivation of Method A predicts, so this check does not support it.")
    return (verdict + f" The sign of the difference is {'the same' if consistent else 'not the same'} in all five subsets of the table. "
            f"This is {images['n']} images with one or two splits each, and Grad-CAM at the last stage has a very coarse grid, so read it as a "
            "direction and a size, not as proof.")


def gradcam_findings(rows: list, plain: str, other: str, device: str) -> str:
    """The `auto:gradcam` section of findings.md: set-up, the numbers for the predicted-class maps and the true-class maps, a reading."""
    all_pairs = energy_stats(rows, plain, other, "energy_pred")
    per_image = energy_stats(per_image_rows(rows, (plain, other)), plain, other, "energy_pred")
    by_part = {part: energy_stats([r for r in rows if r["part"] == part], plain, other, "energy_pred") for part in GRADCAM_PARTS}
    clean = energy_stats([r for r in rows if not r["ambiguous"]], plain, other, "energy_pred")
    true_view = energy_stats(rows, plain, other, "energy_true")
    base, name = gradcam_label(plain), gradcam_label(other)
    checked = [m.reported for r in rows for m in (r[plain], r[other]) if m.reported is not None]
    paragraphs = [
        f"Shortcut check: how much of the Grad-CAM map lies inside the hand-annotated salient leaf. {all_pairs['n']} (image, split) pairs from "
        f"{all_pairs['images']} annotated field images ({by_part['test']['n']} test, {by_part['val']['n']} validation; an image counts in every split where it "
        "was not a training image). The map explains the class the model predicted; it was computed on "
        f"{device} from the retrained checkpoints, whose predictions equal the reported runs' on {sum(checked)} of {len(checked)} pairs checked. "
        "The leaf area share is the chance level: a uniform map would put exactly that share inside the polygon.",
        f"- Leaf area share (chance): {all_pairs['area']:.3f}.\n"
        f"- {base}: energy inside {format_stat(all_pairs[plain])} ({all_pairs[plain][0] / all_pairs['area']:.1f}x chance).\n"
        f"- {name}: energy inside {format_stat(all_pairs[other])} ({all_pairs[other][0] / all_pairs['area']:.1f}x chance).\n"
        f"- Paired difference {name} - {base}: {all_pairs['diff'][0]:+.3f} +- {all_pairs['diff'][1]:.3f}; {name} ahead in {all_pairs['ahead']} of {all_pairs['n']} pairs.",
        "Sub-groups of the same difference: "
        + "; ".join(f"{label} {stats['diff'][0]:+.3f} +- {stats['diff'][1]:.3f} ({stats['ahead']} of {stats['n']})"
                    for label, stats in (("test pairs", by_part["test"]), ("validation pairs", by_part["val"]),
                                         ("without ambiguous images", clean), ("each image once", per_image)))
        + f". With the true class explained instead of the predicted one: {base} {format_stat(true_view[plain])}, {name} {format_stat(true_view[other])}, "
        f"difference {true_view['diff'][0]:+.3f} +- {true_view['diff'][1]:.3f}.",
        "Reading: " + gradcam_reading(rows, plain, other),
    ]
    return "\n\n".join(paragraphs)


def gradcam_tasks(case_picks: dict, pairs: list) -> dict:
    """The images to explain, {(split, filename): {"true", "part", "polygon"}}: the figure images (test, no polygon) and the pairs."""
    tasks = {}
    for picks in case_picks.values():
        for row, split_seed in picks:
            tasks[(split_seed, row["filename"])] = {"true": CLASS_NAMES.index(row["true"]), "part": "test", "polygon": None}
    for pair in pairs:
        tasks[(pair["split"], pair["filename"])] = {"true": CLASS_NAMES.index(pair["cls"]), "part": pair["part"], "polygon": pair["polygon"]}
    return tasks


def gradcam_case_picks(rows: list, split_seed: int | None) -> dict:
    """{category: [(case row, split)]} for the success and failure sheets: the images ``select_cases`` shows on the T7.4 contact
    sheets, each on the first split where it was a test image (only ``split_seed`` when given; images it did not test are left out)."""
    picks = {}
    for category in ("success", "failure"):
        picks[category] = []
        for row in select_cases(rows, category):
            splits = [k for k, pred in zip(ANALYSIS_SPLITS, row["preds"]) if pred and (split_seed is None or k == split_seed)]
            if splits:
                picks[category].append((row, splits[0]))
    return picks


def run_gradcam(data_root, exp_id: str | None, compare_id: str, device: str, split_seed: int | None = None, runs_override=None, out=print) -> list:
    """T7.5: the Grad-CAM figures of the success and failure images, the shortcut check on the annotated field images and its
    findings section. ``exp_id`` is the plain model whose cases are shown (default: the recommended one), ``compare_id`` the second
    model. With ``split_seed`` only that split is used and the files get a `_split<k>` suffix. Returns the files written."""
    data_root = Path(data_root)
    choices = load_choices(data_root)
    exp_id = exp_id or recommended_experiment(choices)
    if not exp_id:
        raise ExperimentError(f"no --exp given and {choices.path} has no 'recommended' experiment (gate G2). Pass --exp ID, for example --exp E2-swin-U")
    exp_ids = (exp_id, compare_id)
    exp, problems = load_experiment(data_root, exp_id, choices)
    if exp is None:
        raise ExperimentError(f"{exp_id} has no complete reported runs ({'; '.join(problems)}), so its cases cannot be selected")
    tag = "" if split_seed is None else f"_split{split_seed}"
    rows = case_rows(pool_predictions(data_root, exp))
    case_picks = gradcam_case_picks(rows, split_seed)
    pairs = annotated_pairs(data_root, splits=ANALYSIS_SPLITS if split_seed is None else (split_seed,))
    tasks = gradcam_tasks(case_picks, pairs)
    missing = sorted({name for _, name in tasks if not (cache_dir(data_root) / name).is_file()})
    if missing:
        raise DatasetError(f"the image cache lacks {len(missing)} images (for example {missing[0]}); run `prepare`")
    measured = {e: measure_model(data_root, choices, e, tasks, device, runs_override, out) for e in dict.fromkeys(exp_ids)}
    written = []
    headings = ["Photo", *[gradcam_label(e) for e in exp_ids]]
    footer = (f"Grad-CAM of the predicted class at the last stage ({GRADCAM_COLORMAP} at {GRADCAM_ALPHA:.0%}; red = most evidence for the class, each map scaled to its own maximum). "
              "Label under a map: true -> predicted (probability of the predicted class) of that model on that split; red = wrong.")
    for category, picks in case_picks.items():
        if picks:
            adjective = {"success": "consistent successes", "failure": "consistent failures"}[category]
            written.append(save_cam_sheet(figures_dir(data_root) / f"gradcam_{category}{tag}.png", case_items(data_root, picks, measured, exp_ids), headings,
                                          f"{exp_id}: Grad-CAM of the {adjective} (test images, one split each)", footer))
    if pairs:
        rows_energy = energy_rows(pairs, measured, exp_ids)
        pairs_by_key = {(p["split"], p["filename"]): p for p in pairs}
        examples = select_annotated_examples(rows_energy)
        written.append(save_cam_sheet(figures_dir(data_root) / f"gradcam_annotated{tag}.png", annotated_items(data_root, pairs_by_key, examples, exp_ids), headings,
                                      "Grad-CAM against the annotated salient leaf (yellow outline), test images", footer.replace("Label under a map", "Under a map: share of the map inside the outline; label")))
        path = results_dir(data_root) / f"gradcam_energy{tag}.csv"
        write_text_atomic(path, energy_csv(rows_energy, exp_ids))
        written.append(path)
        written.extend(write_table(data_root, table_gradcam(rows_energy, exp_id, compare_id, GRADCAM_TABLE_NAME + tag)))
        if split_seed is None:
            findings = results_dir(data_root) / "findings.md"
            existing = read_file_bytes(findings).decode("utf-8") if findings.is_file() else None
            block = auto_block("gradcam", "Grad-CAM shortcut check (T7.5)", gradcam_findings(rows_energy, exp_id, compare_id, device))
            write_text_atomic(findings, update_findings_text(existing, {"gradcam": block}))
            written.append(findings)
    for path in written:
        out(f"gradcam: wrote {path}")
    return written